# Ethereum Bridge Volume Prediction

Experiments predicting ETH trading volume using market data and bridge activity. Run the cells below in order from the repository root. See `README.md` for setup and data details.

In [1]:
import ccxt
import pandas as pd
import numpy as np

from ta import add_all_ta_features
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
horizon = 1
return_threshold = 0.0
start="2023-01-01",
end="2025-12-31",

In [ ]:
# Fetch USD-denominated market data
import yfinance as yf
import pandas as pd
import numpy as np
from ta import add_all_ta_features


from eth_bridge.data import fetch_yfinance_ohlcv_df


gold = fetch_yfinance_ohlcv_df(
    symbol="ETH-USD",
)

data = add_all_ta_features(
    gold.copy(),
    open="open",
    high="high",
    low="low",
    close="close",
    volume="volume",
    fillna=False
)

for col in ["trend_psar_down", "trend_psar_up"]:
    if col in data.columns:
        data.drop(columns=[col], inplace=True)

data["gold_ret_1"] = data["close"].pct_change(1)
data["gold_ret_3"] = data["close"].pct_change(3)
data["gold_ret_5"] = data["close"].pct_change(5)
data["gold_ret_10"] = data["close"].pct_change(10)

data["gold_volatility_5"] = data["gold_ret_1"].rolling(5).std()
data["gold_volatility_10"] = data["gold_ret_1"].rolling(10).std()
data["gold_volatility_20"] = data["gold_ret_1"].rolling(20).std()

data["gold_volume_change"] = data["volume"].pct_change(1)

data["gold_volume_z"] = (
    data["volume"] - data["volume"].rolling(20).mean()
) / data["volume"].rolling(20).std()

data["gold_ma_gap_5"] = data["close"] / data["close"].rolling(5).mean() - 1
data["gold_ma_gap_20"] = data["close"] / data["close"].rolling(20).mean() - 1
data["gold_ma_gap_60"] = data["close"] / data["close"].rolling(60).mean() - 1

data["future_return_d"] = data["close"].shift(-horizon) / data["close"] - 1

data["target"] = np.where(
    data["future_return_d"] > return_threshold,
    1,
    0
)

data = data.replace([np.inf, -np.inf], np.nan)
data.dropna(inplace=True)

group_A = data.iloc[0::3].copy()
group_B = data.iloc[1::3].copy()
group_C = data.iloc[2::3].copy()

data1 = data.copy()

In [4]:
data1

Price,open,high,low,close,volume,volume_adi,volume_obv,volume_cmf,volume_fi,volume_em,...,gold_volatility_5,gold_volatility_10,gold_volatility_20,gold_volume_change,gold_volume_z,gold_ma_gap_5,gold_ma_gap_20,gold_ma_gap_60,future_return_d,target
Date,,,,,,,,,,,,,,,,,,,,,
2023-03-14,1680.295288,1779.917847,1665.902344,1703.507080,16159781303,6.355330e+10,41828888241,0.099376,3.278415e+11,61.324287,...,0.031891,0.038724,0.032241,0.028534,2.027877,0.080099,0.073593,0.063698,-0.027782,0
2023-03-15,1705.189941,1721.079224,1616.630371,1656.180542,12715339905,6.046744e+10,29113548336,0.086191,1.950394e+11,-44.403169,...,0.039304,0.040376,0.032924,-0.213149,0.995246,0.020707,0.043599,0.033013,0.012701,1
2023-03-16,1656.332031,1693.139282,1639.261719,1677.215454,9364980166,6.429667e+10,38478528502,0.124541,1.953183e+11,-1.527041,...,0.039774,0.040385,0.032414,-0.263490,0.087855,0.009458,0.054566,0.044778,0.068727,1
2023-03-17,1677.296753,1799.343994,1667.131836,1792.485107,12467445117,7.547055e+10,50945973619,0.169837,3.727182e+11,71.090437,...,0.038643,0.044449,0.035481,0.331284,0.833844,0.053201,0.120086,0.114088,-0.017197,0
2023-03-18,1793.592407,1840.357544,1751.262817,1761.659668,10317822213,6.756079e+10,40628151406,0.099653,2.740369e+11,54.031353,...,0.037542,0.044398,0.035447,-0.172419,0.196712,0.025288,0.096683,0.092735,0.013576,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-25,2481.777832,2531.084473,2416.602783,2443.159424,17270862024,6.809036e+11,-141061514177,0.179649,1.030878e+12,-3.130542,...,0.044448,0.060295,0.044419,-0.244237,0.455788,-0.009129,0.176727,0.284551,0.025828,1
2026-08-26,2442.489258,2514.666260,2432.512207,2506.260986,13160062688,6.913708e+11,-127901451489,0.221164,1.002241e+12,-0.158810,...,0.025191,0.059284,0.044342,-0.238019,0.004287,0.017208,0.189807,0.307023,0.001656,1
2026-08-27,2506.239502,2564.274414,2481.944824,2510.410156,16865824214,6.861676e+11,-111035627275,0.198345,8.690610e+11,24.173103,...,0.015652,0.059948,0.044397,0.281591,0.349935,0.011815,0.175120,0.298576,-0.027037,0


## 1. Price-direction classification baselines

In [ ]:
#Random Forest baseline

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. SETTINGS
# ============================================================

TRAIN_START = pd.Timestamp("2023-08-01")
TEST_START  = pd.Timestamp("2025-08-01")
TEST_END    = pd.Timestamp("2026-07-31")

TARGET_COL = "target"
FUTURE_RETURN_COL = "future_return_d"

THRESHOLD = 0.50

# Model retraining interval
RETRAIN_DAYS = 7

RANDOM_STATE = 42


# ============================================================
# 2. PREPARE DATA
# ============================================================

# DataFrame with engineered features
data = data1.copy()


# ------------------------------------------------------------
# Date handling
# ------------------------------------------------------------

if data.index.name == "Date":
    data = data.reset_index()


if "Date" in data.columns:

    date_col = "Date"
    data[date_col] = pd.to_datetime(data[date_col])

elif "day" in data.columns:

    date_col = "day"
    data[date_col] = pd.to_datetime(data[date_col])

else:

    raise ValueError(
        "Could not find a Date or day column."
    )


data = (
    data
    .sort_values(date_col)
    .reset_index(drop=True)
)


# ============================================================
# 3. FEATURE LIST
# ============================================================
#
# Features have already been engineered;
# exclude only date, target, and future_return
# ============================================================

exclude_cols = [
    date_col,
    TARGET_COL,
    FUTURE_RETURN_COL
]


feature_cols = [
    col
    for col in data.columns
    if col not in exclude_cols
]


print("=" * 60)
print("FEATURES")
print("=" * 60)

print(
    "Number of features:",
    len(feature_cols)
)


# ============================================================
# 4. NUMERIC CLEANING
# ============================================================

for col in feature_cols:

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )


data[TARGET_COL] = pd.to_numeric(
    data[TARGET_COL],
    errors="coerce"
)


data = data.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# Use only rows with all features present
# ------------------------------------------------------------

data = (
    data
    .dropna(
        subset=
        feature_cols
        +
        [TARGET_COL]
    )
    .reset_index(drop=True)
)


data[TARGET_COL] = (
    data[TARGET_COL]
    .astype(int)
)


# ============================================================
# 5. LIMIT DATA PERIOD
# ============================================================

data = data[
    (data[date_col] >= TRAIN_START)
    &
    (data[date_col] <= TEST_END)
].copy()


data = (
    data
    .sort_values(date_col)
    .reset_index(drop=True)
)


print()
print("=" * 60)
print("DATA PERIOD")
print("=" * 60)

print(
    data[date_col].min(),
    "->",
    data[date_col].max()
)

print(
    "Samples:",
    len(data)
)


# ============================================================
# 6. FIND TEST START INDEX
# ============================================================

test_indices = data.index[
    (data[date_col] >= TEST_START)
    &
    (data[date_col] <= TEST_END)
].tolist()


if len(test_indices) == 0:

    raise ValueError(
        "No data available in the test period."
    )


# ============================================================
# 7. RANDOM FOREST SETTINGS
# ============================================================

RF_PARAMS = {

    "n_estimators": 300,

    "max_depth": 8,

    "min_samples_leaf": 5,

    "max_features": "sqrt",

    "class_weight": "balanced",

    "random_state": RANDOM_STATE,

    "n_jobs": -1
}


# ============================================================
# 8. DAILY FORWARD + WEEKLY RETRAIN
# ============================================================
#
# Important:
#
# Target for each row:
#
# X(t) -> direction(t -> t+1)
#
# The target at time t is not yet known.
#
# The last training sample available at time t is
# the previous row.
#
# Example:
#
# Use features from August 10 to predict August 11
#
# Most recent observable training label:
# 8/9 -> 8/10
#
# ============================================================

predictions = []

model = None

last_retrain_date = None


for idx in test_indices:

    current_row = data.loc[idx]

    current_date = current_row[date_col]


    # ========================================================
    # SHOULD WE RETRAIN?
    # ========================================================

    need_retrain = (

        model is None

        or

        (
            current_date
            -
            last_retrain_date
        ).days
        >=
        RETRAIN_DAYS

    )


    if need_retrain:


        # ====================================================
        # TRAINING DATA
        # ====================================================
        #
        # Use only rows before the current row
        #
        # That is, index < idx
        #
        # The target for row idx-1 is:
        #
        # previous day -> current day
        #
        # This is already known on the current day
        #
        # ====================================================

        train = data.loc[
            : idx - 1
        ].copy()


        # Restrict the training start date
        train = train[
            train[date_col] >= TRAIN_START
        ].copy()


        X_train = (
            train[
                feature_cols
            ]
        )

        y_train = (
            train[
                TARGET_COL
            ]
        )


        # ====================================================
        # NEW RF MODEL
        # ====================================================

        model = RandomForestClassifier(
            **RF_PARAMS
        )


        model.fit(
            X_train,
            y_train
        )


        last_retrain_date = (
            current_date
        )


        print(
            f"Retrain: "
            f"{current_date.date()} | "
            f"Train samples: {len(train)}"
        )


    # ========================================================
    # TODAY'S FEATURE
    # ========================================================

    X_today = (
        data.loc[
            [idx],
            feature_cols
        ]
    )


    # ========================================================
    # PREDICT TOMORROW
    # ========================================================

    prob_up = (
        model.predict_proba(
            X_today
        )[0, 1]
    )


    predicted = int(
        prob_up >= THRESHOLD
    )


    actual = int(
        current_row[
            TARGET_COL
        ]
    )


    predictions.append({

        "feature_date":
            current_date,

        "actual":
            actual,

        "predicted":
            predicted,

        "prob_up":
            prob_up,

        "future_return":
            current_row[
                FUTURE_RETURN_COL
            ]

    })


# ============================================================
# 9. RESULTS
# ============================================================

results = pd.DataFrame(
    predictions
)


y_true = (
    results["actual"]
)

y_pred = (
    results["predicted"]
)

y_prob = (
    results["prob_up"]
)


# ============================================================
# 10. METRICS
# ============================================================

f1_up = f1_score(
    y_true,
    y_pred,
    zero_division=0
)


macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)


accuracy = accuracy_score(
    y_true,
    y_pred
)


balanced_accuracy = (
    balanced_accuracy_score(
        y_true,
        y_pred
    )
)


precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)


recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)


auc = roc_auc_score(
    y_true,
    y_prob
)


# ============================================================
# 11. PRINT RESULTS
# ============================================================

print()
print("=" * 60)
print("ETH WEEKLY-RETRAINED DAILY FORWARD RF")
print("=" * 60)

print(
    f"F1 UP              : {f1_up:.4f}"
)

print(
    f"Macro F1           : {macro_f1:.4f}"
)

print(
    f"Accuracy           : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy:.4f}"
)

print(
    f"Precision UP       : {precision:.4f}"
)

print(
    f"Recall UP          : {recall:.4f}"
)

print(
    f"ROC-AUC            : {auc:.4f}"
)


print()

print(
    "Actual UP ratio   :",
    y_true.mean()
)

print(
    "Predicted UP ratio:",
    y_pred.mean()
)


# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

print()
print("Confusion Matrix")

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)


print()
print(
    classification_report(
        y_true,
        y_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 13. MONTHLY PERFORMANCE
# ============================================================

results["month"] = (
    results[
        "feature_date"
    ]
    .dt.to_period("M")
)


monthly_results = []


for month, g in results.groupby(
    "month"
):

    monthly_results.append({

        "month":
            str(month),

        "n":
            len(g),

        "macro_f1":
            f1_score(
                g["actual"],
                g["predicted"],
                average="macro",
                zero_division=0
            ),

        "f1_up":
            f1_score(
                g["actual"],
                g["predicted"],
                zero_division=0
            ),

        "accuracy":
            accuracy_score(
                g["actual"],
                g["predicted"]
            ),

        "actual_up_ratio":
            g["actual"].mean(),

        "predicted_up_ratio":
            g["predicted"].mean()

    })


monthly_results = pd.DataFrame(
    monthly_results
)


print()
print("=" * 60)
print("MONTHLY RESULTS")
print("=" * 60)

print(
    monthly_results
    .to_string(
        index=False
    )
)


# ============================================================
# 14. SAVE
# ============================================================

results.to_csv(
    "eth_rf_weekly_retrain_predictions.csv",
    index=False
)


monthly_results.to_csv(
    "eth_rf_weekly_retrain_monthly.csv",
    index=False
)

FEATURES
Number of features: 101

DATA PERIOD
2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Samples: 1096
Retrain: 2025-08-01 | Train samples: 731
Retrain: 2025-08-08 | Train samples: 738
Retrain: 2025-08-15 | Train samples: 745
Retrain: 2025-08-22 | Train samples: 752
Retrain: 2025-08-29 | Train samples: 759
Retrain: 2025-09-05 | Train samples: 766
Retrain: 2025-09-12 | Train samples: 773
Retrain: 2025-09-19 | Train samples: 780
Retrain: 2025-09-26 | Train samples: 787
Retrain: 2025-10-03 | Train samples: 794
Retrain: 2025-10-10 | Train samples: 801
Retrain: 2025-10-17 | Train samples: 808
Retrain: 2025-10-24 | Train samples: 815
Retrain: 2025-10-31 | Train samples: 822
Retrain: 2025-11-07 | Train samples: 829
Retrain: 2025-11-14 | Train samples: 836
Retrain: 2025-11-21 | Train samples: 843
Retrain: 2025-11-28 | Train samples: 850
Retrain: 2025-12-05 | Train samples: 857
Retrain: 2025-12-12 | Train samples: 864
Retrain: 2025-12-19 | Train samples: 871
Retrain: 2025-12-26 | Train samples:

In [6]:
print()
print("=" * 60)
print("MONTHLY RESULTS")
print("=" * 60)

print(
    monthly_results
    .to_string(
        index=False
    )
)




MONTHLY RESULTS
  month  n  macro_f1    f1_up  accuracy  actual_up_ratio  predicted_up_ratio
2025-08 31  0.392157 0.117647  0.516129         0.516129            0.032258
2025-09 30  0.562290 0.518519  0.566667         0.500000            0.400000
2025-10 31  0.483333 0.300000  0.548387         0.483871            0.161290
2025-11 30  0.494949 0.545455  0.500000         0.433333            0.666667
2025-12 31  0.449321 0.413793  0.451613         0.516129            0.419355
2026-01 31  0.602564 0.538462  0.612903         0.419355            0.419355
2026-02 28  0.475936 0.588235  0.500000         0.428571            0.785714
2026-03 31  0.470085 0.555556  0.483871         0.580645            0.580645
2026-04 30  0.494949 0.545455  0.500000         0.500000            0.600000
2026-05 31  0.456140 0.578947  0.483871         0.483871            0.741935
2026-06 30  0.499444 0.482759  0.500000         0.400000            0.566667
2026-07 31  0.573545 0.628571  0.580645         0.612903   

In [7]:
monthly_results['macro_f1'].mean()

np.float64(0.4962261745698306)

In [8]:
# Retest with specified Random Forest parameters

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. SETTINGS
# ============================================================

TRAIN_START = pd.Timestamp("2023-08-01")
TEST_START  = pd.Timestamp("2025-08-01")
TEST_END    = pd.Timestamp("2026-07-31")

TARGET_COL = "target"
FUTURE_RETURN_COL = "future_return_d"

THRESHOLD = 0.50

# Model retraining interval
RETRAIN_DAYS = 7

RANDOM_STATE = 42


# ============================================================
# 2. PREPARE DATA
# ============================================================

# DataFrame with engineered features
data = data1.copy()


# ------------------------------------------------------------
# Date handling
# ------------------------------------------------------------

if data.index.name == "Date":
    data = data.reset_index()


if "Date" in data.columns:

    date_col = "Date"
    data[date_col] = pd.to_datetime(data[date_col])

elif "day" in data.columns:

    date_col = "day"
    data[date_col] = pd.to_datetime(data[date_col])

else:

    raise ValueError(
        "Could not find a Date or day column."
    )


data = (
    data
    .sort_values(date_col)
    .reset_index(drop=True)
)


# ============================================================
# 3. FEATURE LIST
# ============================================================
#
# Features have already been engineered;
# exclude only date, target, and future_return
# ============================================================

exclude_cols = [
    date_col,
    TARGET_COL,
    FUTURE_RETURN_COL
]


feature_cols = [
    col
    for col in data.columns
    if col not in exclude_cols
]


print("=" * 60)
print("FEATURES")
print("=" * 60)

print(
    "Number of features:",
    len(feature_cols)
)


# ============================================================
# 4. NUMERIC CLEANING
# ============================================================

for col in feature_cols:

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )


data[TARGET_COL] = pd.to_numeric(
    data[TARGET_COL],
    errors="coerce"
)


data = data.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# Use only rows with all features present
# ------------------------------------------------------------

data = (
    data
    .dropna(
        subset=
        feature_cols
        +
        [TARGET_COL]
    )
    .reset_index(drop=True)
)


data[TARGET_COL] = (
    data[TARGET_COL]
    .astype(int)
)


# ============================================================
# 5. LIMIT DATA PERIOD
# ============================================================

data = data[
    (data[date_col] >= TRAIN_START)
    &
    (data[date_col] <= TEST_END)
].copy()


data = (
    data
    .sort_values(date_col)
    .reset_index(drop=True)
)


print()
print("=" * 60)
print("DATA PERIOD")
print("=" * 60)

print(
    data[date_col].min(),
    "->",
    data[date_col].max()
)

print(
    "Samples:",
    len(data)
)


# ============================================================
# 6. FIND TEST START INDEX
# ============================================================

test_indices = data.index[
    (data[date_col] >= TEST_START)
    &
    (data[date_col] <= TEST_END)
].tolist()


if len(test_indices) == 0:

    raise ValueError(
        "No data available in the test period."
    )


# ============================================================
# 7. RANDOM FOREST SETTINGS
# ============================================================

RF_PARAMS = {

    "n_estimators": 300,

    "max_depth": 8,

    "min_samples_leaf": 5,

    "max_features": "sqrt",

    "class_weight": "balanced",

    "random_state": RANDOM_STATE,

    "n_jobs": -1
}


# ============================================================
# 8. DAILY FORWARD + WEEKLY RETRAIN
# ============================================================
#
# Important:
#
# Target for each row:
#
# X(t) -> direction(t -> t+1)
#
# The target at time t is not yet known.
#
# The last training sample available at time t is
# the previous row.
#
# Example:
#
# Use features from August 10 to predict August 11
#
# Most recent observable training label:
# 8/9 -> 8/10
#
# ============================================================

predictions = []

model = None

last_retrain_date = None


for idx in test_indices:

    current_row = data.loc[idx]

    current_date = current_row[date_col]


    # ========================================================
    # SHOULD WE RETRAIN?
    # ========================================================

    need_retrain = (

        model is None

        or

        (
            current_date
            -
            last_retrain_date
        ).days
        >=
        RETRAIN_DAYS

    )


    if need_retrain:


        # ====================================================
        # TRAINING DATA
        # ====================================================
        #
        # Use only rows before the current row
        #
        # That is, index < idx
        #
        # The target for row idx-1 is:
        #
        # previous day -> current day
        #
        # This is already known on the current day
        #
        # ====================================================

        train = data.loc[
            : idx - 1
        ].copy()


        # Restrict the training start date
        train = train[
            train[date_col] >= TRAIN_START
        ].copy()


        X_train = (
            train[
                feature_cols
            ]
        )

        y_train = (
            train[
                TARGET_COL
            ]
        )


        # ====================================================
        # NEW RF MODEL
        # ====================================================

        model = RandomForestClassifier(
            **RF_PARAMS
        )


        model.fit(
            X_train,
            y_train
        )


        last_retrain_date = (
            current_date
        )


        print(
            f"Retrain: "
            f"{current_date.date()} | "
            f"Train samples: {len(train)}"
        )


    # ========================================================
    # TODAY'S FEATURE
    # ========================================================

    X_today = (
        data.loc[
            [idx],
            feature_cols
        ]
    )


    # ========================================================
    # PREDICT TOMORROW
    # ========================================================

    prob_up = (
        model.predict_proba(
            X_today
        )[0, 1]
    )


    predicted = int(
        prob_up >= THRESHOLD
    )


    actual = int(
        current_row[
            TARGET_COL
        ]
    )


    predictions.append({

        "feature_date":
            current_date,

        "actual":
            actual,

        "predicted":
            predicted,

        "prob_up":
            prob_up,

        "future_return":
            current_row[
                FUTURE_RETURN_COL
            ]

    })


# ============================================================
# 9. RESULTS
# ============================================================

results = pd.DataFrame(
    predictions
)


y_true = (
    results["actual"]
)

y_pred = (
    results["predicted"]
)

y_prob = (
    results["prob_up"]
)


# ============================================================
# 10. METRICS
# ============================================================

f1_up = f1_score(
    y_true,
    y_pred,
    zero_division=0
)


macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)


accuracy = accuracy_score(
    y_true,
    y_pred
)


balanced_accuracy = (
    balanced_accuracy_score(
        y_true,
        y_pred
    )
)


precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)


recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)


auc = roc_auc_score(
    y_true,
    y_prob
)


# ============================================================
# 11. PRINT RESULTS
# ============================================================

print()
print("=" * 60)
print("ETH WEEKLY-RETRAINED DAILY FORWARD RF")
print("=" * 60)

print(
    f"F1 UP              : {f1_up:.4f}"
)

print(
    f"Macro F1           : {macro_f1:.4f}"
)

print(
    f"Accuracy           : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy:.4f}"
)

print(
    f"Precision UP       : {precision:.4f}"
)

print(
    f"Recall UP          : {recall:.4f}"
)

print(
    f"ROC-AUC            : {auc:.4f}"
)


print()

print(
    "Actual UP ratio   :",
    y_true.mean()
)

print(
    "Predicted UP ratio:",
    y_pred.mean()
)


# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

print()
print("Confusion Matrix")

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)


print()
print(
    classification_report(
        y_true,
        y_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 13. MONTHLY PERFORMANCE
# ============================================================

results["month"] = (
    results[
        "feature_date"
    ]
    .dt.to_period("M")
)


monthly_results = []


for month, g in results.groupby(
    "month"
):

    monthly_results.append({

        "month":
            str(month),

        "n":
            len(g),

        "macro_f1":
            f1_score(
                g["actual"],
                g["predicted"],
                average="macro",
                zero_division=0
            ),

        "f1_up":
            f1_score(
                g["actual"],
                g["predicted"],
                zero_division=0
            ),

        "accuracy":
            accuracy_score(
                g["actual"],
                g["predicted"]
            ),

        "actual_up_ratio":
            g["actual"].mean(),

        "predicted_up_ratio":
            g["predicted"].mean()

    })


monthly_results = pd.DataFrame(
    monthly_results
)


print()
print("=" * 60)
print("MONTHLY RESULTS")
print("=" * 60)

print(
    monthly_results
    .to_string(
        index=False
    )
)


# ============================================================
# 14. SAVE
# ============================================================

results.to_csv(
    "eth_rf_weekly_retrain_predictions.csv",
    index=False
)


monthly_results.to_csv(
    "eth_rf_weekly_retrain_monthly.csv",
    index=False
)

FEATURES
Number of features: 101

DATA PERIOD
2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Samples: 1096
Retrain: 2025-08-01 | Train samples: 731
Retrain: 2025-08-08 | Train samples: 738
Retrain: 2025-08-15 | Train samples: 745
Retrain: 2025-08-22 | Train samples: 752
Retrain: 2025-08-29 | Train samples: 759
Retrain: 2025-09-05 | Train samples: 766
Retrain: 2025-09-12 | Train samples: 773
Retrain: 2025-09-19 | Train samples: 780
Retrain: 2025-09-26 | Train samples: 787
Retrain: 2025-10-03 | Train samples: 794
Retrain: 2025-10-10 | Train samples: 801
Retrain: 2025-10-17 | Train samples: 808
Retrain: 2025-10-24 | Train samples: 815
Retrain: 2025-10-31 | Train samples: 822
Retrain: 2025-11-07 | Train samples: 829
Retrain: 2025-11-14 | Train samples: 836
Retrain: 2025-11-21 | Train samples: 843
Retrain: 2025-11-28 | Train samples: 850
Retrain: 2025-12-05 | Train samples: 857
Retrain: 2025-12-12 | Train samples: 864
Retrain: 2025-12-19 | Train samples: 871
Retrain: 2025-12-26 | Train samples:

In [9]:
#Check mean F1
monthly_results['macro_f1'].mean()

np.float64(0.4962261745698306)

In [11]:
# Check data1: downloaded ETH-USD data
data1

Price,open,high,low,close,volume,volume_adi,volume_obv,volume_cmf,volume_fi,volume_em,...,gold_volatility_5,gold_volatility_10,gold_volatility_20,gold_volume_change,gold_volume_z,gold_ma_gap_5,gold_ma_gap_20,gold_ma_gap_60,future_return_d,target
Date,,,,,,,,,,,,,,,,,,,,,
2023-03-14,1680.295288,1779.917847,1665.902344,1703.507080,16159781303,6.355330e+10,41828888241,0.099376,3.278415e+11,61.324287,...,0.031891,0.038724,0.032241,0.028534,2.027877,0.080099,0.073593,0.063698,-0.027782,0
2023-03-15,1705.189941,1721.079224,1616.630371,1656.180542,12715339905,6.046744e+10,29113548336,0.086191,1.950394e+11,-44.403169,...,0.039304,0.040376,0.032924,-0.213149,0.995246,0.020707,0.043599,0.033013,0.012701,1
2023-03-16,1656.332031,1693.139282,1639.261719,1677.215454,9364980166,6.429667e+10,38478528502,0.124541,1.953183e+11,-1.527041,...,0.039774,0.040385,0.032414,-0.263490,0.087855,0.009458,0.054566,0.044778,0.068727,1
2023-03-17,1677.296753,1799.343994,1667.131836,1792.485107,12467445117,7.547055e+10,50945973619,0.169837,3.727182e+11,71.090437,...,0.038643,0.044449,0.035481,0.331284,0.833844,0.053201,0.120086,0.114088,-0.017197,0
2023-03-18,1793.592407,1840.357544,1751.262817,1761.659668,10317822213,6.756079e+10,40628151406,0.099653,2.740369e+11,54.031353,...,0.037542,0.044398,0.035447,-0.172419,0.196712,0.025288,0.096683,0.092735,0.013576,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-25,2481.777832,2531.084473,2416.602783,2443.159424,17270862024,6.809036e+11,-141061514177,0.179649,1.030878e+12,-3.130542,...,0.044448,0.060295,0.044419,-0.244237,0.455788,-0.009129,0.176727,0.284551,0.025828,1
2026-08-26,2442.489258,2514.666260,2432.512207,2506.260986,13160062688,6.913708e+11,-127901451489,0.221164,1.002241e+12,-0.158810,...,0.025191,0.059284,0.044342,-0.238019,0.004287,0.017208,0.189807,0.307023,0.001656,1
2026-08-27,2506.239502,2564.274414,2481.944824,2510.410156,16865824214,6.861676e+11,-111035627275,0.198345,8.690610e+11,24.173103,...,0.015652,0.059948,0.044397,0.281591,0.349935,0.011815,0.175120,0.298576,-0.027037,0


In [13]:
# Load raw bridge data
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# =========================================================
# 1. LOAD DATA
# =========================================================

file_path = "bridge_raw_data.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.shape)
print(df["l2_chain"].unique())


# =========================================================
# 2. BASIC CLEANING
# =========================================================

df["day"] = pd.to_datetime(df["day"], errors="coerce")

# invalid dates remove
df = df.dropna(subset=["day"]).copy()

numeric_cols = [
    "ethereum_to_l2_usd",
    "l2_to_ethereum_usd",
    "ethereum_net_flow_usd",
    "gross_bridge_volume_usd",
    "out_tx_count",
    "in_tx_count",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df = df.sort_values(["day", "l2_chain"]).reset_index(drop=True)


# 3. DAILY AGGREGATED BRIDGE DATA
# =========================================================

daily = (
    df.groupby("day", as_index=False)
      .agg(
          total_outflow_usd=("ethereum_to_l2_usd", "sum"),
          total_inflow_usd=("l2_to_ethereum_usd", "sum"),
          net_flow_usd=("ethereum_net_flow_usd", "sum"),
          gross_volume_usd=("gross_bridge_volume_usd", "sum"),
          out_tx_count=("out_tx_count", "sum"),
          in_tx_count=("in_tx_count", "sum"),
      )
)

daily["total_tx_count"] = (
    daily["out_tx_count"] +
    daily["in_tx_count"]
)

# 4. BRIDGE IMBALANCE INDEX (BII)
# =========================================================
#
# +1  = strong flow toward Ethereum
# -1  = strong flow from Ethereum to L2
#  0  = balanced
#

denominator = (
    daily["total_inflow_usd"] +
    daily["total_outflow_usd"]
)

daily["BII"] = np.where(
    denominator > 0,
    (
        daily["total_inflow_usd"]
        - daily["total_outflow_usd"]
    ) / denominator,
    0
)


# =========================================================
# 5. TRANSACTION IMBALANCE
# =========================================================

tx_denominator = (
    daily["in_tx_count"] +
    daily["out_tx_count"]
)

daily["tx_imbalance"] = np.where(
    tx_denominator > 0,
    (
        daily["in_tx_count"]
        - daily["out_tx_count"]
    ) / tx_denominator,
    0
)

# =========================================================
# 6. AVERAGE TRANSFER SIZE (ATS)
# =========================================================

daily["avg_transfer_size_usd"] = np.where(
    daily["total_tx_count"] > 0,
    daily["gross_volume_usd"] /
    daily["total_tx_count"],
    0
)



# =========================================================
# 7. LOG-TRANSFORMED ACTIVITY
# =========================================================

daily["log_bridge_activity"] = np.log1p(
    daily["gross_volume_usd"]
)

daily["log_tx_activity"] = np.log1p(
    daily["total_tx_count"]
)

daily["log_avg_transfer_size"] = np.log1p(
    daily["avg_transfer_size_usd"]
)


# =========================================================
# 8. CHAIN SHARES
# =========================================================

volume_wide = (
    df.pivot_table(
        index="day",
        columns="l2_chain",
        values="gross_bridge_volume_usd",
        aggfunc="sum",
        fill_value=0
    )
)

# ensure expected chains exist
chains = [
    "arbitrum",
    "base",
    "optimism",
    "polygon",
    "zksync"
]

for chain in chains:
    if chain not in volume_wide.columns:
        volume_wide[chain] = 0

volume_wide = volume_wide[chains]

total_volume = volume_wide.sum(axis=1)

share = volume_wide.div(
    total_volume.replace(0, np.nan),
    axis=0
).fillna(0)

# =========================================================
# 9. BRIDGE CONCENTRATION INDEX (HHI)
# =========================================================
#
# higher = concentrated on few L2s
# lower  = evenly distributed
#

HHI = (share ** 2).sum(axis=1)

HHI.name = "bridge_HHI"


# =========================================================
# 10. BRIDGE DIVERSIFICATION INDEX (ENTROPY)
# =========================================================
#
# normalized 0 ~ 1
#
# 0 = concentrated
# 1 = evenly diversified
#

safe_share = share.replace(0, np.nan)

entropy = -(safe_share * np.log(safe_share)).sum(axis=1)

normalized_entropy = entropy / np.log(len(chains))

normalized_entropy = normalized_entropy.fillna(0)

normalized_entropy.name = "bridge_entropy"


# =========================================================
# 11. L2 ROTATION INDEX (LRI)
# =========================================================
#
# Measures day-to-day changes in L2 allocation.
#
# 0 = same allocation as yesterday
# 1 = very strong rotation
#

rotation = (
    share.diff()
         .abs()
         .sum(axis=1)
         / 2
)

rotation = rotation.fillna(0)

rotation.name = "L2_rotation_index"


# =========================================================
# 12. MERGE STRUCTURAL INDICATORS
# =========================================================

structural = pd.concat(
    [
        HHI,
        normalized_entropy,
        rotation
    ],
    axis=1
).reset_index()

daily = daily.merge(
    structural,
    on="day",
    how="left"
)



# =========================================================
# 13. ADD EACH L2 SHARE
# =========================================================

share.columns = [
    f"{c}_volume_share"
    for c in share.columns
]

share = share.reset_index()

daily = daily.merge(
    share,
    on="day",
    how="left"
)


# =========================================================
# 14. FLOW MOMENTUM
# =========================================================

daily = daily.sort_values("day")

daily["net_flow_3d"] = (
    daily["net_flow_usd"]
    .rolling(3)
    .sum()
)

daily["net_flow_7d"] = (
    daily["net_flow_usd"]
    .rolling(7)
    .sum()
)

daily["net_flow_30d"] = (
    daily["net_flow_usd"]
    .rolling(30)
    .sum()
)


daily["bridge_volume_ma7"] = (
    daily["gross_volume_usd"]
    .rolling(7)
    .mean()
)

daily["bridge_volume_ma30"] = (
    daily["gross_volume_usd"]
    .rolling(30)
    .mean()
)


# volume momentum
daily["bridge_volume_momentum_7d"] = (
    daily["gross_volume_usd"]
    / daily["bridge_volume_ma7"]
    - 1
)


# =========================================================
# 15. ABNORMAL BRIDGE FLOW
# =========================================================
#
# Current net flow relative to previous 30-day behavior
#

rolling_mean = (
    daily["net_flow_usd"]
    .rolling(30)
    .mean()
)

rolling_std = (
    daily["net_flow_usd"]
    .rolling(30)
    .std()
)

daily["abnormal_net_flow"] = (
    daily["net_flow_usd"] - rolling_mean
) / rolling_std.replace(0, np.nan)

# =========================================================
# 16. BRIDGE ACTIVITY Z-SCORES
# =========================================================
#
# Rolling z-score avoids using future information.
#

from eth_bridge.data import rolling_zscore


daily["activity_z"] = rolling_zscore(
    daily["log_bridge_activity"], 90
)

daily["tx_activity_z"] = rolling_zscore(
    daily["log_tx_activity"], 90
)

daily["transfer_size_z"] = rolling_zscore(
    daily["log_avg_transfer_size"], 90
)

# =========================================================
# 17. SIMPLE ECONOMIC BRIDGE ACTIVITY INDEX
# =========================================================
#
# Direction (BII) is intentionally NOT included.
# This index measures intensity/activity.
#

daily["CBI_simple"] = (
      0.40 * daily["activity_z"]
    + 0.25 * daily["tx_activity_z"]
    + 0.15 * daily["transfer_size_z"]
    + 0.10 * daily["bridge_entropy"]
    + 0.10 * daily["L2_rotation_index"]
)



# =========================================================
# 19. FINAL FEATURES
# =========================================================

final_features = [
    "day",

    # raw
    "total_outflow_usd",
    "total_inflow_usd",
    "net_flow_usd",
    "gross_volume_usd",
    "total_tx_count",

    # economic indicators
    "BII",
    "tx_imbalance",
    "avg_transfer_size_usd",

    # structural indicators
    "bridge_HHI",
    "bridge_entropy",
    "L2_rotation_index",

    # chain allocation
    "arbitrum_volume_share",
    "base_volume_share",
    "optimism_volume_share",
    "polygon_volume_share",
    "zksync_volume_share",

    # momentum / abnormal activity
    "net_flow_3d",
    "net_flow_7d",
    "net_flow_30d",
    "bridge_volume_momentum_7d",
    "abnormal_net_flow",

    # composite indices
    # "CBI_simple",
    # "CBI_PCA",
]

bridge_features = daily[
    final_features
].copy()


          day l2_chain  ethereum_to_l2_usd  l2_to_ethereum_usd  \
0  2020-08-26  polygon            0.000000              0.0000   
1  2020-08-28  polygon          197.747500              1.9428   
2  2020-08-29  polygon            0.100006              0.0000   
3  2020-09-01  polygon         2312.186418              0.0000   
4  2020-09-02  polygon         2190.950000              0.0000   

   ethereum_net_flow_usd  gross_bridge_volume_usd  out_tx_count  in_tx_count  
0               0.000000                 0.000000             1            0  
1            -195.804700               199.690300             3            1  
2              -0.100006                 0.100006             2            0  
3           -2312.186418              2312.186418             4            0  
4           -2190.950000              2190.950000             1            0  
(8490, 8)
<StringArray>
['polygon', 'arbitrum', 'zksync', 'optimism', 'base']
Length: 5, dtype: str


In [16]:
# Features and the daily DataFrame were prepared above

In [17]:
# Save bridge indicators
# =========================================================
# 20. SAVE
# =========================================================

bridge_features.to_csv(
    "bridge_indicators2.csv",
    index=False
)

print(bridge_features.tail())

            day  total_outflow_usd  total_inflow_usd  net_flow_usd  \
2161 2026-07-31       1.119987e+07      6.877360e+06 -4.322510e+06   
2162 2026-08-01       6.883562e+06      8.730492e+06  1.846931e+06   
2163 2026-08-02       8.499307e+06      8.435754e+06 -6.355326e+04   
2164 2026-08-03       1.571153e+07      1.052555e+07 -5.185978e+06   
2165 2026-08-04       1.605929e+05      1.572358e+04 -1.448694e+05   

      gross_volume_usd  total_tx_count       BII  tx_imbalance  \
2161      1.807723e+07            5076 -0.239114     -0.127660   
2162      1.561405e+07            4908  0.118286      0.059495   
2163      1.693506e+07            4656 -0.003753      0.035653   
2164      2.623708e+07            5094 -0.197658     -0.076561   
2165      1.763165e+05              92 -0.821644     -0.086957   

      avg_transfer_size_usd  bridge_HHI  ...  arbitrum_volume_share  \
2161            3561.314189    0.404039  ...               0.557589   
2162            3181.347573    0.537324 

In [19]:
# Inspect features selected from daily in bridge_features
bridge_features

,day,total_outflow_usd,total_inflow_usd,net_flow_usd,gross_volume_usd,total_tx_count,BII,tx_imbalance,avg_transfer_size_usd,bridge_HHI,...,arbitrum_volume_share,base_volume_share,optimism_volume_share,polygon_volume_share,zksync_volume_share,net_flow_3d,net_flow_7d,net_flow_30d,bridge_volume_momentum_7d,abnormal_net_flow
0,2020-08-26,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1,0.000000,-1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
1,2020-08-28,1.977475e+02,1.942800e+00,-1.958047e+02,1.996903e+02,4,-0.980542,-0.500000,49.922575,1.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2,2020-08-29,1.000059e-01,0.000000e+00,-1.000059e-01,1.000059e-01,2,-1.000000,-1.000000,0.050003,1.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,-1.959047e+02,NaN,NaN,NaN,NaN
3,2020-09-01,2.312186e+03,0.000000e+00,-2.312186e+03,2.312186e+03,4,-1.000000,-1.000000,578.046604,1.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,-2.508091e+03,NaN,NaN,NaN,NaN
4,2020-09-02,2.190950e+03,0.000000e+00,-2.190950e+03,2.190950e+03,1,-1.000000,-1.000000,2190.950000,1.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,-4.503236e+03,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2161,2026-07-31,1.119987e+07,6.877360e+06,-4.322510e+06,1.807723e+07,5076,-0.239114,-0.127660,3561.314189,0.404039,...,0.557589,0.218618,0.010926,0.212649,0.000217,-5.823925e+06,-1.616705e+07,-2.757427e+08,-0.647667,0.384413
2162,2026-08-01,6.883562e+06,8.730492e+06,1.846931e+06,1.561405e+07,4908,0.118286,0.059495,3181.347573,0.537324,...,0.699607,0.193475,0.004069,0.102100,0.000750,-1.219916e+05,-1.578117e+07,-2.718591e+08,-0.699987,0.854864
2163,2026-08-02,8.499307e+06,8.435754e+06,-6.355326e+04,1.693506e+07,4656,-0.003753,0.035653,3637.255288,0.335083,...,0.411327,0.273935,0.013200,0.301128,0.000410,-2.539133e+06,-1.475575e+07,-2.477769e+08,-0.682750,0.653831
2164,2026-08-03,1.571153e+07,1.052555e+07,-5.185978e+06,2.623708e+07,5094,-0.197658,-0.076561,5150.585649,0.397931,...,0.546576,0.283819,0.036873,0.131419,0.001313,-3.402600e+06,-1.636675e+07,-2.488159e+08,-0.518619,0.248144


In [20]:
# Separate raw and engineered features
# ============================================================
# BRIDGE FEATURE GROUPS
# ============================================================

# ------------------------------------------------------------
# 1. RAW BRIDGE FEATURES
# ------------------------------------------------------------

raw_bridge_features = [
    "total_outflow_usd",
    "total_inflow_usd",
    "net_flow_usd",
    "gross_volume_usd",
    "total_tx_count"
]


# ------------------------------------------------------------
# 2. ENGINEERED BRIDGE FEATURES
# ------------------------------------------------------------

engineered_bridge_features = [
    "BII",
    "tx_imbalance",
    "avg_transfer_size_usd",
    "bridge_HHI",

    "arbitrum_volume_share",
    "base_volume_share",
    "optimism_volume_share",
    "polygon_volume_share",
    "zksync_volume_share",

    "net_flow_3d",
    "net_flow_7d",
    "net_flow_30d",

    "bridge_volume_momentum_7d",
    "abnormal_net_flow"
]


# ============================================================
# CREATE TWO DATAFRAMES
# ============================================================

bridge_raw = bridge_features[
    ["day"] + raw_bridge_features
].copy()


bridge_engineered = bridge_features[
    ["day"] + engineered_bridge_features
].copy()


# ============================================================
# CHECK
# ============================================================

print("RAW BRIDGE")
print(bridge_raw.shape)
print(bridge_raw.head())

print()

print("ENGINEERED BRIDGE")
print(bridge_engineered.shape)
print(bridge_engineered.head())

RAW BRIDGE
(2166, 6)
         day  total_outflow_usd  total_inflow_usd  net_flow_usd  \
0 2020-08-26           0.000000            0.0000      0.000000   
1 2020-08-28         197.747500            1.9428   -195.804700   
2 2020-08-29           0.100006            0.0000     -0.100006   
3 2020-09-01        2312.186418            0.0000  -2312.186418   
4 2020-09-02        2190.950000            0.0000  -2190.950000   

   gross_volume_usd  total_tx_count  
0          0.000000               1  
1        199.690300               4  
2          0.100006               2  
3       2312.186418               4  
4       2190.950000               1  

ENGINEERED BRIDGE
(2166, 15)
         day       BII  tx_imbalance  avg_transfer_size_usd  bridge_HHI  \
0 2020-08-26  0.000000          -1.0               0.000000         0.0   
1 2020-08-28 -0.980542          -0.5              49.922575         1.0   
2 2020-08-29 -1.000000          -1.0               0.050003         1.0   
3 2020-09-01 -1.00

In [22]:
from pathlib import Path

Path("bridges").mkdir(parents=True, exist_ok=True)

# Save features
bridge_raw.to_csv('bridges/bridge_raw.csv', index=False)
bridge_engineered.to_csv('bridges/bridge_engineered.csv', index=False)

In [23]:
# Set up M1–M4

# ============================================================
# 1. PREPARE MARKET DATA
# ============================================================

market = data1.copy()

# Move Date from the index to a column if needed
if market.index.name == "Date":
    market = market.reset_index()

market["Date"] = pd.to_datetime(market["Date"])


# ============================================================
# 2. PREPARE BRIDGE DATA
# ============================================================

bridge_raw["day"] = pd.to_datetime(
    bridge_raw["day"]
)

bridge_engineered["day"] = pd.to_datetime(
    bridge_engineered["day"]
)


# ============================================================
# 3. MODEL 1: MARKET ONLY
# ============================================================

data_m1 = market.copy()


# ============================================================
# 4. MODEL 2: MARKET + RAW BRIDGE
# ============================================================

data_m2 = market.merge(
    bridge_raw,
    left_on="Date",
    right_on="day",
    how="inner"
)

data_m2 = data_m2.drop(
    columns=["day"]
)


# ============================================================
# 5. MODEL 3: MARKET + ENGINEERED BRIDGE
# ============================================================

data_m3 = market.merge(
    bridge_engineered,
    left_on="Date",
    right_on="day",
    how="inner"
)

data_m3 = data_m3.drop(
    columns=["day"]
)


# ============================================================
# 6. MODEL 4: MARKET + RAW + ENGINEERED BRIDGE
# ============================================================

bridge_all = bridge_raw.merge(
    bridge_engineered,
    on="day",
    how="inner"
)

data_m4 = market.merge(
    bridge_all,
    left_on="Date",
    right_on="day",
    how="inner"
)

data_m4 = data_m4.drop(
    columns=["day"]
)


# ============================================================
# 7. CHECK
# ============================================================

for name, d in {
    "M1 Market": data_m1,
    "M2 Market + Raw Bridge": data_m2,
    "M3 Market + Engineered Bridge": data_m3,
    "M4 Market + All Bridge": data_m4
}.items():

    print(
        name,
        "| rows:", len(d),
        "| cols:", len(d.columns),
        "|",
        d["Date"].min(),
        "->",
        d["Date"].max()
    )

M1 Market | rows: 1265 | cols: 104 | 2023-03-14 00:00:00 -> 2026-08-29 00:00:00
M2 Market + Raw Bridge | rows: 1240 | cols: 109 | 2023-03-14 00:00:00 -> 2026-08-04 00:00:00
M3 Market + Engineered Bridge | rows: 1240 | cols: 118 | 2023-03-14 00:00:00 -> 2026-08-04 00:00:00
M4 Market + All Bridge | rows: 1240 | cols: 123 | 2023-03-14 00:00:00 -> 2026-08-04 00:00:00


In [24]:
# Check
data_m3.head()

,Date,open,high,low,close,volume,volume_adi,volume_obv,volume_cmf,volume_fi,...,arbitrum_volume_share,base_volume_share,optimism_volume_share,polygon_volume_share,zksync_volume_share,net_flow_3d,net_flow_7d,net_flow_30d,bridge_volume_momentum_7d,abnormal_net_flow
0,2023-03-14,1680.295288,1779.917847,1665.902344,1703.507080,16159781303,6.355330e+10,41828888241,0.099376,3.278415e+11,...,0.778347,0.0,0.055204,0.165053,0.001396,-6.179928e+07,-1.222734e+08,-6.718163e+08,-0.170270,0.193531
1,2023-03-15,1705.189941,1721.079224,1616.630371,1656.180542,12715339905,6.046744e+10,29113548336,0.086191,1.950394e+11,...,0.610028,0.0,0.005118,0.384383,0.000472,-8.721560e+07,-1.471104e+08,-6.865802e+08,0.110870,-0.741126
2,2023-03-16,1656.332031,1693.139282,1639.261719,1677.215454,9364980166,6.429667e+10,38478528502,0.124541,1.953183e+11,...,0.856478,0.0,0.014025,0.128459,0.001038,-7.345268e+07,-1.710189e+08,-6.837040e+08,0.483535,0.236842
3,2023-03-17,1677.296753,1799.343994,1667.131836,1792.485107,12467445117,7.547055e+10,50945973619,0.169837,3.727182e+11,...,0.849189,0.0,0.016214,0.129563,0.005033,-8.249594e+07,-1.749106e+08,-6.814590e+08,0.004706,-0.314827
4,2023-03-18,1793.592407,1840.357544,1751.262817,1761.659668,10317822213,6.756079e+10,40628151406,0.099653,2.740369e+11,...,0.622029,0.0,0.005573,0.364868,0.007530,-6.509070e+07,-1.625449e+08,-6.531955e+08,-0.192975,0.211858


In [25]:
data1.head()

Price,open,high,low,close,volume,volume_adi,volume_obv,volume_cmf,volume_fi,volume_em,...,gold_volatility_5,gold_volatility_10,gold_volatility_20,gold_volume_change,gold_volume_z,gold_ma_gap_5,gold_ma_gap_20,gold_ma_gap_60,future_return_d,target
Date,,,,,,,,,,,,,,,,,,,,,
2023-03-14,1680.295288,1779.917847,1665.902344,1703.507080,16159781303,6.355330e+10,41828888241,0.099376,3.278415e+11,61.324287,...,0.031891,0.038724,0.032241,0.028534,2.027877,0.080099,0.073593,0.063698,-0.027782,0
2023-03-15,1705.189941,1721.079224,1616.630371,1656.180542,12715339905,6.046744e+10,29113548336,0.086191,1.950394e+11,-44.403169,...,0.039304,0.040376,0.032924,-0.213149,0.995246,0.020707,0.043599,0.033013,0.012701,1
2023-03-16,1656.332031,1693.139282,1639.261719,1677.215454,9364980166,6.429667e+10,38478528502,0.124541,1.953183e+11,-1.527041,...,0.039774,0.040385,0.032414,-0.263490,0.087855,0.009458,0.054566,0.044778,0.068727,1
2023-03-17,1677.296753,1799.343994,1667.131836,1792.485107,12467445117,7.547055e+10,50945973619,0.169837,3.727182e+11,71.090437,...,0.038643,0.044449,0.035481,0.331284,0.833844,0.053201,0.120086,0.114088,-0.017197,0
2023-03-18,1793.592407,1840.357544,1751.262817,1761.659668,10317822213,6.756079e+10,40628151406,0.099653,2.740369e+11,54.031353,...,0.037542,0.044398,0.035447,-0.172419,0.196712,0.025288,0.096683,0.092735,0.013576,1


In [33]:

# Random Forest evaluation for the feature sets

# ============================================================
# 1. SETTINGS
# ============================================================

TRAIN_START = pd.Timestamp("2023-08-01")
TEST_START  = pd.Timestamp("2025-08-01")
TEST_END    = pd.Timestamp("2026-07-31")

TARGET_COL = "target"
FUTURE_RETURN_COL = "future_return_d"

THRESHOLD = 0.50
# Model retraining interval
RETRAIN_DAYS = 7
RANDOM_STATE = 42


# ============================================================
# 2. PREPARE DATA
# ============================================================

# DataFrame with engineered features

def rf_test(data_mx, nametag):
    data = data_mx.copy()


    # ------------------------------------------------------------
    # Date handling
    # ------------------------------------------------------------

    if data.index.name == "Date":
        data = data.reset_index()


    if "Date" in data.columns:

        date_col = "Date"
        data[date_col] = pd.to_datetime(data[date_col])

    elif "day" in data.columns:

        date_col = "day"
        data[date_col] = pd.to_datetime(data[date_col])

    else:

        raise ValueError(
            "Could not find a Date or day column."
        )


    data = (
        data
        .sort_values(date_col)
        .reset_index(drop=True)
    )


    # ============================================================
    # 3. FEATURE LIST
    # ============================================================
    #
    # Features have already been engineered;
    # exclude only date, target, and future_return
    # ============================================================

    exclude_cols = [
        date_col,
        TARGET_COL,
        FUTURE_RETURN_COL
    ]


    feature_cols = [
        col
        for col in data.columns
        if col not in exclude_cols
    ]


    print("=" * 60)
    print("FEATURES")
    print("=" * 60)

    print(
        "Number of features:",
        len(feature_cols)
    )


    # ============================================================
    # 4. NUMERIC CLEANING
    # ============================================================

    for col in feature_cols:

        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )


    data[TARGET_COL] = pd.to_numeric(
        data[TARGET_COL],
        errors="coerce"
    )


    data = data.replace(
        [np.inf, -np.inf],
        np.nan
    )


    # ------------------------------------------------------------
    # Use only rows with all features present
    # ------------------------------------------------------------

    data = (
        data
        .dropna(
            subset=
            feature_cols
            +
            [TARGET_COL]
        )
        .reset_index(drop=True)
    )


    data[TARGET_COL] = (
        data[TARGET_COL]
        .astype(int)
    )


    # ============================================================
    # 5. LIMIT DATA PERIOD
    # ============================================================

    data = data[
        (data[date_col] >= TRAIN_START)
        &
        (data[date_col] <= TEST_END)
    ].copy()


    data = (
        data
        .sort_values(date_col)
        .reset_index(drop=True)
    )


    print()
    print("=" * 60)
    print("DATA PERIOD")
    print("=" * 60)

    print(
        data[date_col].min(),
        "->",
        data[date_col].max()
    )

    print(
        "Samples:",
        len(data)
    )


    # ============================================================
    # 6. FIND TEST START INDEX
    # ============================================================

    test_indices = data.index[
        (data[date_col] >= TEST_START)
        &
        (data[date_col] <= TEST_END)
    ].tolist()


    if len(test_indices) == 0:

        raise ValueError(
            "No data available in the test period."
        )


    # ============================================================
    # 7. RANDOM FOREST SETTINGS
    # ============================================================

    RF_PARAMS = {

        "n_estimators": 300,

        "max_depth": 8,

        "min_samples_leaf": 5,

        "max_features": "sqrt",

        "class_weight": "balanced",

        "random_state": RANDOM_STATE,

        "n_jobs": -1
    }


    # ============================================================
    # 8. DAILY FORWARD + WEEKLY RETRAIN
    # ============================================================
    #
    # Important:
    #
    # Target for each row:
    #
    # X(t) -> direction(t -> t+1)
    #
    # The target at time t is not yet known.
    #
    # The last training sample available at time t is
    # the previous row.
    #
    # Example:
    #
    # Use features from August 10 to predict August 11
    #
    # Most recent observable training label:
    # 8/9 -> 8/10
    #
    # ============================================================

    predictions = []

    model = None

    last_retrain_date = None


    for idx in test_indices:

        current_row = data.loc[idx]

        current_date = current_row[date_col]


        # ========================================================
        # SHOULD WE RETRAIN?
        # ========================================================

        need_retrain = (

            model is None

            or

            (
                current_date
                -
                last_retrain_date
            ).days
            >=
            RETRAIN_DAYS

        )


        if need_retrain:


            # ====================================================
            # TRAINING DATA
            # ====================================================
            #
            # Use only rows before the current row
            #
            # That is, index < idx
            #
            # The target for row idx-1 is:
            #
            # previous day -> current day
            #
            # This is already known on the current day
            #
            # ====================================================

            train = data.loc[
                : idx - 1
            ].copy()


            # Restrict the training start date
            train = train[
                train[date_col] >= TRAIN_START
            ].copy()


            X_train = (
                train[
                    feature_cols
                ]
            )

            y_train = (
                train[
                    TARGET_COL
                ]
            )


            # ====================================================
            # NEW RF MODEL
            # ====================================================

            model = RandomForestClassifier(
                **RF_PARAMS
            )


            model.fit(
                X_train,
                y_train
            )


            last_retrain_date = (
                current_date
            )


            print(
                f"Retrain: "
                f"{current_date.date()} | "
                f"Train samples: {len(train)}"
            )


        # ========================================================
        # TODAY'S FEATURE
        # ========================================================

        X_today = (
            data.loc[
                [idx],
                feature_cols
            ]
        )


        # ========================================================
        # PREDICT TOMORROW
        # ========================================================

        prob_up = (
            model.predict_proba(
                X_today
            )[0, 1]
        )


        predicted = int(
            prob_up >= THRESHOLD
        )


        actual = int(
            current_row[
                TARGET_COL
            ]
        )


        predictions.append({

            "feature_date":
                current_date,

            "actual":
                actual,

            "predicted":
                predicted,

            "prob_up":
                prob_up,

            "future_return":
                current_row[
                    FUTURE_RETURN_COL
                ]

        })


    # ============================================================
    # 9. RESULTS
    # ============================================================

    results = pd.DataFrame(
        predictions
    )


    y_true = (
        results["actual"]
    )

    y_pred = (
        results["predicted"]
    )

    y_prob = (
        results["prob_up"]
    )


    # ============================================================
    # 10. METRICS
    # ============================================================

    f1_up = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    auc = roc_auc_score(
        y_true,
        y_prob
    )


    # ============================================================
    # 11. PRINT RESULTS
    # ============================================================

    print()
    print("=" * 60)
    print("ETH WEEKLY-RETRAINED DAILY FORWARD RF")
    print("=" * 60)

    print(
        f"F1 UP              : {f1_up:.4f}"
    )

    print(
        f"Macro F1           : {macro_f1:.4f}"
    )

    print(
        f"Accuracy           : {accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy  : {balanced_accuracy:.4f}"
    )

    print(
        f"Precision UP       : {precision:.4f}"
    )

    print(
        f"Recall UP          : {recall:.4f}"
    )

    print(
        f"ROC-AUC            : {auc:.4f}"
    )


    print()

    print(
        "Actual UP ratio   :",
        y_true.mean()
    )

    print(
        "Predicted UP ratio:",
        y_pred.mean()
    )


    # ============================================================
    # 12. CONFUSION MATRIX
    # ============================================================

    print()
    print("Confusion Matrix")

    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )


    print()
    print(
        classification_report(
            y_true,
            y_pred,
            digits=4,
            zero_division=0
        )
    )


    # ============================================================
    # 13. MONTHLY PERFORMANCE
    # ============================================================

    results["month"] = (
        results[
            "feature_date"
        ]
        .dt.to_period("M")
    )


    monthly_results = []


    for month, g in results.groupby(
        "month"
    ):

        monthly_results.append({

            "month":
                str(month),

            "n":
                len(g),

            "macro_f1":
                f1_score(
                    g["actual"],
                    g["predicted"],
                    average="macro",
                    zero_division=0
                ),

            "f1_up":
                f1_score(
                    g["actual"],
                    g["predicted"],
                    zero_division=0
                ),

            "accuracy":
                accuracy_score(
                    g["actual"],
                    g["predicted"]
                ),

            "actual_up_ratio":
                g["actual"].mean(),

            "predicted_up_ratio":
                g["predicted"].mean()

        })


    monthly_results = pd.DataFrame(
        monthly_results
    )


    # print(f"{nametag} MONTHLY RESULTS")
    # print("=" * 60)
    # print("MONTHLY RESULTS")
    # print("=" * 60)

    # print(
    #     monthly_results
    #     .to_string(
    #         index=False
    #     )
    # )


    # ============================================================
    # 14. SAVE
    # ============================================================

    # results.to_csv(
    #     f"{nametag}_eth_rf_weekly_retrain_predictions.csv",
    #     index=False
    # )


    # monthly_results.to_csv(
    #     f"{nametag}_eth_rf_weekly_retrain_monthly.csv",
    #     index=False
    # )

    return monthly_results

In [37]:
monthly_results_m1 = rf_test(data_m1, "M1")
monthly_results_m2 = rf_test(data_m2, "M2")
monthly_results_m3 = rf_test(data_m3, "M3")
monthly_results_m4 = rf_test(data_m4, "M4")

FEATURES
Number of features: 101

DATA PERIOD
2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Samples: 1096
Retrain: 2025-08-01 | Train samples: 731
Retrain: 2025-08-08 | Train samples: 738
Retrain: 2025-08-15 | Train samples: 745
Retrain: 2025-08-22 | Train samples: 752
Retrain: 2025-08-29 | Train samples: 759
Retrain: 2025-09-05 | Train samples: 766
Retrain: 2025-09-12 | Train samples: 773
Retrain: 2025-09-19 | Train samples: 780
Retrain: 2025-09-26 | Train samples: 787
Retrain: 2025-10-03 | Train samples: 794
Retrain: 2025-10-10 | Train samples: 801
Retrain: 2025-10-17 | Train samples: 808
Retrain: 2025-10-24 | Train samples: 815
Retrain: 2025-10-31 | Train samples: 822
Retrain: 2025-11-07 | Train samples: 829
Retrain: 2025-11-14 | Train samples: 836
Retrain: 2025-11-21 | Train samples: 843
Retrain: 2025-11-28 | Train samples: 850
Retrain: 2025-12-05 | Train samples: 857
Retrain: 2025-12-12 | Train samples: 864
Retrain: 2025-12-19 | Train samples: 871
Retrain: 2025-12-26 | Train samples:

In [38]:
# data_m1
monthly_results_m1['macro_f1'].mean(), monthly_results_m1['accuracy'].mean()

(np.float64(0.4962261745698306), np.float64(0.5203405017921148))

In [44]:
# data_m2
monthly_results_m2['macro_f1'].mean(), monthly_results_m2['accuracy'].mean()

(np.float64(0.47069580220656543), np.float64(0.4954301075268817))

In [40]:
# data_m3
monthly_results_m3['macro_f1'].mean(), monthly_results_m3['accuracy'].mean()

(np.float64(0.45097851826612256), np.float64(0.4842997951868919))

In [41]:
# data_m4
monthly_results_m4['macro_f1'].mean(), monthly_results_m4['accuracy'].mean()

(np.float64(0.4648146394637815), np.float64(0.4870775729646697))

## 2. Volume forecasts: M1–M4 and forecast horizons

In [42]:
# Review bridge_features
bridge_only_features = [
    "total_outflow_usd",
    "total_inflow_usd",
    "net_flow_usd",
    "gross_volume_usd",
    "total_tx_count",

    "BII",
    "tx_imbalance",
    "avg_transfer_size_usd",
    "bridge_HHI",

    "arbitrum_volume_share",
    "base_volume_share",
    "optimism_volume_share",
    "polygon_volume_share",
    "zksync_volume_share",

    "net_flow_3d",
    "net_flow_7d",
    "net_flow_30d",

    "bridge_volume_momentum_7d",
    "abnormal_net_flow"
]

feature_cols = bridge_only_features

In [43]:
# Select feature columns from data_m4
data_m4[feature_cols]

,total_outflow_usd,total_inflow_usd,net_flow_usd,gross_volume_usd,total_tx_count,BII,tx_imbalance,avg_transfer_size_usd,bridge_HHI,arbitrum_volume_share,base_volume_share,optimism_volume_share,polygon_volume_share,zksync_volume_share,net_flow_3d,net_flow_7d,net_flow_30d,bridge_volume_momentum_7d,abnormal_net_flow
0,2.486713e+07,5.775240e+06,-1.909188e+07,3.064237e+07,2679,-0.623055,-0.824561,11437.986409,0.636117,0.778347,0.000000,0.055204,0.165053,0.001396,-6.179928e+07,-1.222734e+08,-6.718163e+08,-0.170270,0.193531
1,4.030717e+07,4.652273e+06,-3.565490e+07,4.495945e+07,2127,-0.793046,-0.777151,21137.492785,0.519911,0.610028,0.000000,0.005118,0.384383,0.000472,-8.721560e+07,-1.471104e+08,-6.865802e+08,0.110870,-0.741126
2,4.235435e+07,2.364845e+07,-1.870590e+07,6.600280e+07,3085,-0.283411,-0.843760,21394.747928,0.750254,0.856478,0.000000,0.014025,0.128459,0.001038,-7.345268e+07,-1.710189e+08,-6.837040e+08,0.483535,0.236842
3,3.738453e+07,9.249389e+06,-2.813514e+07,4.663392e+07,4763,-0.603319,-0.882427,9790.871498,0.738197,0.849189,0.000000,0.016214,0.129563,0.005033,-8.249594e+07,-1.749106e+08,-6.814590e+08,0.004706,-0.314827
4,2.577631e+07,7.526648e+06,-1.824966e+07,3.330296e+07,5747,-0.547989,-0.908822,5794.842653,0.520137,0.622029,0.000000,0.005573,0.364868,0.007530,-6.509070e+07,-1.625449e+08,-6.531955e+08,-0.192975,0.211858
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1235,1.119987e+07,6.877360e+06,-4.322510e+06,1.807723e+07,5076,-0.239114,-0.127660,3561.314189,0.404039,0.557589,0.218618,0.010926,0.212649,0.000217,-5.823925e+06,-1.616705e+07,-2.757427e+08,-0.647667,0.384413
1236,6.883562e+06,8.730492e+06,1.846931e+06,1.561405e+07,4908,0.118286,0.059495,3181.347573,0.537324,0.699607,0.193475,0.004069,0.102100,0.000750,-1.219916e+05,-1.578117e+07,-2.718591e+08,-0.699987,0.854864
1237,8.499307e+06,8.435754e+06,-6.355326e+04,1.693506e+07,4656,-0.003753,0.035653,3637.255288,0.335083,0.411327,0.273935,0.013200,0.301128,0.000410,-2.539133e+06,-1.475575e+07,-2.477769e+08,-0.682750,0.653831
1238,1.571153e+07,1.052555e+07,-5.185978e+06,2.623708e+07,5094,-0.197658,-0.076561,5150.585649,0.397931,0.546576,0.283819,0.036873,0.131419,0.001313,-3.402600e+06,-1.636675e+07,-2.488159e+08,-0.518619,0.248144


In [45]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# VOLUME PREDICTION FUNCTION
# ============================================================

from eth_bridge.forecasting import test_volume


In [50]:
results_h1_m1, monthly_results_h1_m1, summary1_h1_m1 = test_volume(
    data_m1,
    horizon=1,
    model_name="M1 Market"
)

results_h1_m2, monthly_results_h1_m2, summary2_h1_m2 = test_volume(
    data_m2,
    horizon=1,
    model_name="M2 Market + Raw Bridge"
)

results_h1_m3, monthly_results_h1_m3, summary3_h1_m3 = test_volume(
    data_m3,
    horizon=1,
    model_name="M3 Market + Engineered Bridge"
)

results_h1_m4, monthly_results_h1_m4, summary4_h1_m4 = test_volume(
    data_m4,
    horizon=1,
    model_name="M4 Market + All Bridge"
)


M1 Market | ETH Volume t+1
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-31 | n=731
Retrain: 2025-08-08 | Train through: 2025-08-07 | n=738
Retrain: 2025-08-15 | Train through: 2025-08-14 | n=745
Retrain: 2025-08-22 | Train through: 2025-08-21 | n=752
Retrain: 2025-08-29 | Train through: 2025-08-28 | n=759
Retrain: 2025-09-05 | Train through: 2025-09-04 | n=766
Retrain: 2025-09-12 | Train through: 2025-09-11 | n=773
Retrain: 2025-09-19 | Train through: 2025-09-18 | n=780
Retrain: 2025-09-26 | Train through: 2025-09-25 | n=787
Retrain: 2025-10-03 | Train through: 2025-10-02 | n=794
Retrain: 2025-10-10 | Train through: 2025-10-09 | n=801
Retrain: 2025-10-17 | Train through: 2025-10-16 | n=808
Retrain: 2025-10-24 | Train through: 2025-10-23 | n=815
Retrain: 2025-10-31 | Train through: 2025-10-30 | n=822
Retrain: 2025-11-07 | Train through: 2025-11-06 | n=829
Retrain: 2025-11-14 | Train through: 2025-11-13 | n=

In [51]:
# Horizon = 3
results_h3_m1, monthly_results_h3_m1, summary1_h3_m1 = test_volume(
    data_m1,
    horizon=3,
    model_name="M1 Market"
)

results_h3_m2, monthly_results_h3_m2, summary2_h3_m2 = test_volume(
    data_m2,
    horizon=3,
    model_name="M2 Market + Raw Bridge"
)

results_h3_m3, monthly_results_h3_m3, summary3_h3_m3 = test_volume(
    data_m3,
    horizon=3,
    model_name="M3 Market + Engineered Bridge"
)

results_h3_m4, monthly_results_h3_m4, summary4_h3_m4 = test_volume(
    data_m4,
    horizon=3,
    model_name="M4 Market + All Bridge"
)


M1 Market | ETH Volume t+3
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-29 | n=729
Retrain: 2025-08-08 | Train through: 2025-08-05 | n=736
Retrain: 2025-08-15 | Train through: 2025-08-12 | n=743
Retrain: 2025-08-22 | Train through: 2025-08-19 | n=750
Retrain: 2025-08-29 | Train through: 2025-08-26 | n=757
Retrain: 2025-09-05 | Train through: 2025-09-02 | n=764
Retrain: 2025-09-12 | Train through: 2025-09-09 | n=771
Retrain: 2025-09-19 | Train through: 2025-09-16 | n=778
Retrain: 2025-09-26 | Train through: 2025-09-23 | n=785
Retrain: 2025-10-03 | Train through: 2025-09-30 | n=792
Retrain: 2025-10-10 | Train through: 2025-10-07 | n=799
Retrain: 2025-10-17 | Train through: 2025-10-14 | n=806
Retrain: 2025-10-24 | Train through: 2025-10-21 | n=813
Retrain: 2025-10-31 | Train through: 2025-10-28 | n=820
Retrain: 2025-11-07 | Train through: 2025-11-04 | n=827
Retrain: 2025-11-14 | Train through: 2025-11-11 | n=

In [52]:
# Horizon = 7
results_h7_m1, monthly_results_h7_m1, summary1_h7_m1 = test_volume(
    data_m1,
    horizon=7,
    model_name="M1 Market"
)

results_h7_m2, monthly_results_h7_m2, summary2_h7_m2 = test_volume(
    data_m2,
    horizon=7,
    model_name="M2 Market + Raw Bridge"
)

results_h7_m3, monthly_results_h7_m3, summary3_h7_m3 = test_volume(
    data_m3,
    horizon=7,
    model_name="M3 Market + Engineered Bridge"
)

results_h7_m4, monthly_results_h7_m4, summary4_h7_m4 = test_volume(
    data_m4,
    horizon=7,
    model_name="M4 Market + All Bridge"
)


M1 Market | ETH Volume t+7
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-25 | n=725
Retrain: 2025-08-08 | Train through: 2025-08-01 | n=732
Retrain: 2025-08-15 | Train through: 2025-08-08 | n=739
Retrain: 2025-08-22 | Train through: 2025-08-15 | n=746
Retrain: 2025-08-29 | Train through: 2025-08-22 | n=753
Retrain: 2025-09-05 | Train through: 2025-08-29 | n=760
Retrain: 2025-09-12 | Train through: 2025-09-05 | n=767
Retrain: 2025-09-19 | Train through: 2025-09-12 | n=774
Retrain: 2025-09-26 | Train through: 2025-09-19 | n=781
Retrain: 2025-10-03 | Train through: 2025-09-26 | n=788
Retrain: 2025-10-10 | Train through: 2025-10-03 | n=795
Retrain: 2025-10-17 | Train through: 2025-10-10 | n=802
Retrain: 2025-10-24 | Train through: 2025-10-17 | n=809
Retrain: 2025-10-31 | Train through: 2025-10-24 | n=816
Retrain: 2025-11-07 | Train through: 2025-10-31 | n=823
Retrain: 2025-11-14 | Train through: 2025-11-07 | n=

In [53]:
pd.DataFrame([summary1_h1_m1, summary2_h1_m2, summary3_h1_m3, summary4_h1_m4,
                summary1_h3_m1, summary2_h3_m2, summary3_h3_m3, summary4_h3_m4,
                summary1_h7_m1, summary2_h7_m2, summary3_h7_m3, summary4_h7_m4])

,model,horizon,n_features,n_test,mae,rmse,r2,direction_accuracy
0,M1 Market,1,101,365,0.278944,0.362972,0.248362,0.663014
1,M2 Market + Raw Bridge,1,106,365,0.276576,0.362633,0.249767,0.676712
2,M3 Market + Engineered Bridge,1,115,365,0.276496,0.362945,0.248475,0.693151
3,M4 Market + All Bridge,1,120,365,0.275329,0.362912,0.248612,0.698630
4,M1 Market,3,101,365,0.333059,0.418955,0.398476,0.734247
5,M2 Market + Raw Bridge,3,106,365,0.332171,0.419238,0.397662,0.728767
6,M3 Market + Engineered Bridge,3,115,365,0.329394,0.416090,0.406674,0.734247
7,M4 Market + All Bridge,3,120,365,0.329979,0.417301,0.403217,0.717808
8,M1 Market,7,101,365,0.281541,0.362241,0.218230,0.624658
9,M2 Market + Raw Bridge,7,106,362,0.281372,0.364190,0.215775,0.638122


In [54]:
# make dataframes for summaries

summary_df_h1 = pd.DataFrame([
    summary1_h1_m1,
    summary2_h1_m2,
    summary3_h1_m3,
    summary4_h1_m4
])

summary_df_h3 = pd.DataFrame([
    summary1_h3_m1,
    summary2_h3_m2,
    summary3_h3_m3,
    summary4_h3_m4
])

summary_df_h7 = pd.DataFrame([
    summary1_h7_m1,
    summary2_h7_m2,
    summary3_h7_m3,
    summary4_h7_m4
])

In [55]:
# ============================================================
# 1. ORGANIZE RESULTS
# ============================================================

results_dict = {
    1: {
        "M1": results_h1_m1,
        "M2": results_h1_m2,
        "M3": results_h1_m3,
        "M4": results_h1_m4,
    },

    3: {
        "M1": results_h3_m1,
        "M2": results_h3_m2,
        "M3": results_h3_m3,
        "M4": results_h3_m4,
    },

    7: {
        "M1": results_h7_m1,
        "M2": results_h7_m2,
        "M3": results_h7_m3,
        "M4": results_h7_m4,
    }
}

In [56]:
# date valid check
# ============================================================
# 2. COMMON-DATE COMPARISON
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd


comparison_results = []


for horizon, models in results_dict.items():

    # --------------------------------------------------------
    # Find common dates across M1 ~ M4
    # --------------------------------------------------------

    common_dates = None

    for model_name, df in models.items():

        dates = set(
            pd.to_datetime(
                df["feature_date"]
            )
        )

        if common_dates is None:
            common_dates = dates
        else:
            common_dates = (
                common_dates.intersection(dates)
            )


    common_dates = sorted(common_dates)

    print(
        f"Horizon t+{horizon}: "
        f"{len(common_dates)} common dates"
    )


    # --------------------------------------------------------
    # Evaluate each model using SAME dates
    # --------------------------------------------------------

    for model_name, df in models.items():

        temp = df.copy()

        temp["feature_date"] = pd.to_datetime(
            temp["feature_date"]
        )


        temp = temp[
            temp["feature_date"].isin(
                common_dates
            )
        ].copy()


        temp = temp.sort_values(
            "feature_date"
        )


        y_true = temp[
            "actual_volume_change"
        ]

        y_pred = temp[
            "predicted_volume_change"
        ]


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        mae = mean_absolute_error(
            y_true,
            y_pred
        )


        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )


        r2 = r2_score(
            y_true,
            y_pred
        )


        # Direction accuracy

        actual_direction = (
            y_true > 0
        ).astype(int)

        predicted_direction = (
            y_pred > 0
        ).astype(int)


        direction_accuracy = (
            actual_direction.values
            ==
            predicted_direction.values
        ).mean()


        comparison_results.append({

            "horizon":
                horizon,

            "model":
                model_name,

            "n":
                len(temp),

            "mae":
                mae,

            "rmse":
                rmse,

            "r2":
                r2,

            "direction_accuracy":
                direction_accuracy
        })


# ============================================================
# 3. FINAL TABLE
# ============================================================

comparison_common = pd.DataFrame(
    comparison_results
)

comparison_common

Horizon t+1: 365 common dates
Horizon t+3: 365 common dates
Horizon t+7: 362 common dates


,horizon,model,n,mae,rmse,r2,direction_accuracy
0,1,M1,365,0.278944,0.362972,0.248362,0.663014
1,1,M2,365,0.276576,0.362633,0.249767,0.676712
2,1,M3,365,0.276496,0.362945,0.248475,0.693151
3,1,M4,365,0.275329,0.362912,0.248612,0.698630
4,3,M1,365,0.333059,0.418955,0.398476,0.734247
5,3,M2,365,0.332171,0.419238,0.397662,0.728767
6,3,M3,365,0.329394,0.416090,0.406674,0.734247
7,3,M4,365,0.329979,0.417301,0.403217,0.717808
8,7,M1,362,0.282689,0.363491,0.218784,0.627072
9,7,M2,362,0.281372,0.364190,0.215775,0.638122


In [58]:
# Compare model R² scores in a pivot table
comparison_common.pivot(
    index="model",
    columns="horizon",
    values="r2"
)

horizon,1,3,7
model,,,
M1,0.248362,0.398476,0.218784
M2,0.249767,0.397662,0.215775
M3,0.248475,0.406674,0.209297
M4,0.248612,0.403217,0.214139


In [59]:
comparison_common.pivot(
    index="model",
    columns="horizon",
    values="mae"
)

horizon,1,3,7
model,,,
M1,0.278944,0.333059,0.282689
M2,0.276576,0.332171,0.281372
M3,0.276496,0.329394,0.284000
M4,0.275329,0.329979,0.282826


In [60]:
comparison_common.pivot(
    index="model",
    columns="horizon",
    values="direction_accuracy"
)

horizon,1,3,7
model,,,
M1,0.663014,0.734247,0.627072
M2,0.676712,0.728767,0.638122
M3,0.693151,0.734247,0.627072
M4,0.698630,0.717808,0.624309


## 3. Forecast error comparisons and Diebold–Mariano tests

In [ ]:
# value check
# Example: t+3

check = (
    results_h3_m1[
        ["feature_date", "actual_volume_change"]
    ]
    .merge(
        results_h3_m3[
            ["feature_date", "actual_volume_change"]
        ],
        on="feature_date",
        suffixes=("_m1", "_m3")
    )
)

np.allclose(
    check["actual_volume_change_m1"],
    check["actual_volume_change_m3"]
)

True

In [63]:
# Compare M1 and M3 results
dm_data = (
    results_h3_m1[
        [
            "feature_date",
            "actual_volume_change",
            "predicted_volume_change"
        ]
    ]
    .merge(
        results_h3_m3[
            [
                "feature_date",
                "predicted_volume_change"
            ]
        ],
        on="feature_date",
        suffixes=("_m1", "_m3")
    )
)

dm_data["error_m1"] = (
    dm_data["actual_volume_change"]
    - dm_data["predicted_volume_change_m1"]
)

dm_data["error_m3"] = (
    dm_data["actual_volume_change"]
    - dm_data["predicted_volume_change_m3"]
)

dm_data[
    [
        "feature_date",
        "error_m1",
        "error_m3"
    ]
].head()

,feature_date,error_m1,error_m3
0,2025-08-01,-0.178256,-0.148432
1,2025-08-02,0.021797,0.065176
2,2025-08-03,0.048945,0.041680
3,2025-08-04,0.231255,0.268199
4,2025-08-05,0.340569,0.267645


In [64]:
import numpy as np
from scipy.stats import norm


# ============================================================
# DIEBOLD-MARIANO TEST
# M1 Market vs M3 Market + Engineered Bridge
# Horizon = 3
# Loss = Squared Error
# HAC / Newey-West variance
# ============================================================

def dm_test_hac(error1, error2, horizon=3):

    e1 = np.asarray(error1)
    e2 = np.asarray(error2)

    # --------------------------------------------------------
    # 1. Loss differential
    #
    # d > 0:
    # M1 error > M3 error
    # → M3 is better
    # --------------------------------------------------------

    d = e1**2 - e2**2

    d = d[np.isfinite(d)]

    T = len(d)

    mean_d = np.mean(d)


    # --------------------------------------------------------
    # 2. HAC / Newey-West long-run variance
    #
    # For h-step overlapping forecasts:
    # lag = h - 1
    # --------------------------------------------------------

    max_lag = horizon - 1

    centered = d - mean_d


    # lag 0 covariance
    gamma0 = np.sum(
        centered * centered
    ) / T


    long_run_variance = gamma0


    # Newey-West weighted autocovariances
    for lag in range(1, max_lag + 1):

        gamma = np.sum(
            centered[lag:]
            *
            centered[:-lag]
        ) / T

        weight = 1 - lag / (max_lag + 1)

        long_run_variance += (
            2 * weight * gamma
        )


    # --------------------------------------------------------
    # 3. DM statistic
    # --------------------------------------------------------

    variance_mean = (
        long_run_variance / T
    )

    dm_stat = (
        mean_d
        /
        np.sqrt(variance_mean)
    )


    # --------------------------------------------------------
    # 4. Two-sided p-value
    # --------------------------------------------------------

    p_value = (
        2
        *
        (
            1
            -
            norm.cdf(
                abs(dm_stat)
            )
        )
    )


    return {
        "n": T,
        "mean_loss_difference": mean_d,
        "dm_stat": dm_stat,
        "p_value": p_value
    }


# ============================================================
# RUN
# ============================================================

dm_result = dm_test_hac(

    dm_data["error_m1"],

    dm_data["error_m3"],

    horizon=3
)


dm_result

{'n': 365,
 'mean_loss_difference': np.float64(0.002392054845908575),
 'dm_stat': np.float64(0.8040686182023415),
 'p_value': np.float64(0.42135734720935325)}

In [65]:
import numpy as np
import pandas as pd
from scipy.stats import norm


# ============================================================
# 1. SETTINGS
# ============================================================

HORIZON = 3

engineered_bridge_features = [
    "BII",
    "tx_imbalance",
    "avg_transfer_size_usd",
    "bridge_HHI",

    "arbitrum_volume_share",
    "base_volume_share",
    "optimism_volume_share",
    "polygon_volume_share",
    "zksync_volume_share",

    "net_flow_3d",
    "net_flow_7d",
    "net_flow_30d",

    "bridge_volume_momentum_7d",
    "abnormal_net_flow"
]


# ============================================================
# 2. DM TEST WITH HAC / NEWey-WEST
# ============================================================
#
# error1 = baseline M1 error
# error2 = M1 + bridge indicator error
#
# mean_loss_difference > 0:
# candidate model has lower squared error than baseline
# ============================================================

def dm_test_hac(
    error1,
    error2,
    horizon=3
):

    e1 = np.asarray(
        error1,
        dtype=float
    )

    e2 = np.asarray(
        error2,
        dtype=float
    )


    # squared-error loss difference
    d = (
        e1 ** 2
        -
        e2 ** 2
    )


    valid = (
        np.isfinite(d)
    )

    d = d[valid]


    T = len(d)

    mean_d = np.mean(d)


    centered = (
        d
        -
        mean_d
    )


    # --------------------------------------------------------
    # HAC long-run variance
    # --------------------------------------------------------

    gamma0 = (
        np.sum(
            centered * centered
        )
        /
        T
    )


    long_run_variance = (
        gamma0
    )


    max_lag = (
        horizon - 1
    )


    for lag in range(
        1,
        max_lag + 1
    ):

        gamma = (

            np.sum(
                centered[lag:]
                *
                centered[:-lag]
            )

            /

            T
        )


        weight = (
            1
            -
            lag
            /
            (max_lag + 1)
        )


        long_run_variance += (
            2
            *
            weight
            *
            gamma
        )


    # --------------------------------------------------------
    # Safety
    # --------------------------------------------------------

    if long_run_variance <= 0:

        return {
            "mean_loss_difference":
                mean_d,

            "dm_stat":
                np.nan,

            "p_value":
                np.nan
        }


    variance_mean = (
        long_run_variance
        /
        T
    )


    dm_stat = (
        mean_d
        /
        np.sqrt(
            variance_mean
        )
    )


    p_value = (

        2

        *

        (
            1
            -
            norm.cdf(
                abs(dm_stat)
            )
        )

    )


    return {

        "mean_loss_difference":
            mean_d,

        "dm_stat":
            dm_stat,

        "p_value":
            p_value
    }


# ============================================================
# 3. BASELINE M1
# ============================================================

baseline_results, baseline_monthly, baseline_summary = (
    test_volume(
        data_m1,
        horizon=HORIZON,
        model_name="M1 Market"
    )
)


baseline_r2 = (
    baseline_summary["r2"]
)

baseline_rmse = (
    baseline_summary["rmse"]
)

baseline_mae = (
    baseline_summary["mae"]
)


print()
print("=" * 70)
print("BASELINE")
print("=" * 70)

print(
    baseline_summary
)


# ============================================================
# 4. TEST EACH ENGINEERED BRIDGE FEATURE
# ============================================================

indicator_results = []


for bridge_feature in engineered_bridge_features:


    print()
    print("=" * 70)

    print(
        "Testing:",
        bridge_feature
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Check feature exists
    # --------------------------------------------------------

    if bridge_feature not in data_m3.columns:

        print(
            f"Skip: {bridge_feature} not found"
        )

        continue


    # ========================================================
    # 4.1 CREATE M1 + ONE BRIDGE FEATURE
    # ========================================================

    candidate_data = (
        data_m1.copy()
    )


    # --------------------------------------------------------
    # Date handling for M1
    # --------------------------------------------------------

    if candidate_data.index.name == "Date":

        candidate_data = (
            candidate_data
            .reset_index()
        )


    if "Date" not in candidate_data.columns:

        raise ValueError(
            "Missing Date in data_m1."
        )


    candidate_data["Date"] = (
        pd.to_datetime(
            candidate_data["Date"]
        )
    )


    # --------------------------------------------------------
    # Prepare bridge feature
    # --------------------------------------------------------

    bridge_temp = (
        data_m3.copy()
    )


    if bridge_temp.index.name == "Date":

        bridge_temp = (
            bridge_temp
            .reset_index()
        )


    bridge_temp["Date"] = (
        pd.to_datetime(
            bridge_temp["Date"]
        )
    )


    bridge_temp = (
        bridge_temp[
            [
                "Date",
                bridge_feature
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Merge exactly one bridge feature
    # --------------------------------------------------------

    candidate_data = (
        candidate_data
        .merge(
            bridge_temp,
            on="Date",
            how="inner"
        )
    )


    # ========================================================
    # 4.2 RUN FORECAST
    # ========================================================

    candidate_results, candidate_monthly, candidate_summary = (
        test_volume(
            candidate_data,
            horizon=HORIZON,
            model_name=f"M1 + {bridge_feature}"
        )
    )


    # ========================================================
    # 4.3 ALIGN BASELINE AND CANDIDATE DATES
    # ========================================================

    base_temp = (
        baseline_results[
            [
                "feature_date",
                "actual_volume_change",
                "predicted_volume_change"
            ]
        ]
        .copy()
    )


    candidate_temp = (
        candidate_results[
            [
                "feature_date",
                "actual_volume_change",
                "predicted_volume_change"
            ]
        ]
        .copy()
    )


    base_temp["feature_date"] = (
        pd.to_datetime(
            base_temp["feature_date"]
        )
    )


    candidate_temp["feature_date"] = (
        pd.to_datetime(
            candidate_temp["feature_date"]
        )
    )


    compare = (
        base_temp
        .merge(
            candidate_temp,
            on="feature_date",
            suffixes=(
                "_baseline",
                "_candidate"
            )
        )
    )


    # ========================================================
    # 4.4 CHECK ACTUAL TARGET
    # ========================================================

    same_actual = (
        np.allclose(
            compare[
                "actual_volume_change_baseline"
            ],
            compare[
                "actual_volume_change_candidate"
            ]
        )
    )


    if not same_actual:

        raise ValueError(
            f"Actual target mismatch: "
            f"{bridge_feature}"
        )


    # ========================================================
    # 4.5 CALCULATE ERRORS
    # ========================================================

    compare["error_baseline"] = (

        compare[
            "actual_volume_change_baseline"
        ]

        -

        compare[
            "predicted_volume_change_baseline"
        ]

    )


    compare["error_candidate"] = (

        compare[
            "actual_volume_change_candidate"
        ]

        -

        compare[
            "predicted_volume_change_candidate"
        ]

    )


    # ========================================================
    # 4.6 DM TEST
    # ========================================================

    dm = dm_test_hac(

        compare["error_baseline"],

        compare["error_candidate"],

        horizon=HORIZON
    )


    # ========================================================
    # 4.7 PERFORMANCE DIFFERENCE
    # ========================================================

    delta_r2 = (
        candidate_summary["r2"]
        -
        baseline_r2
    )


    delta_rmse = (
        candidate_summary["rmse"]
        -
        baseline_rmse
    )


    delta_mae = (
        candidate_summary["mae"]
        -
        baseline_mae
    )


    # ========================================================
    # 4.8 SAVE RESULT
    # ========================================================

    indicator_results.append({

        "feature":
            bridge_feature,

        "n":
            len(compare),

        "mae":
            candidate_summary["mae"],

        "delta_mae":
            delta_mae,

        "rmse":
            candidate_summary["rmse"],

        "delta_rmse":
            delta_rmse,

        "r2":
            candidate_summary["r2"],

        "delta_r2":
            delta_r2,

        "direction_accuracy":
            candidate_summary[
                "direction_accuracy"
            ],

        "mean_loss_difference":
            dm[
                "mean_loss_difference"
            ],

        "dm_stat":
            dm[
                "dm_stat"
            ],

        "dm_p_value":
            dm[
                "p_value"
            ]

    })


# ============================================================
# 5. FINAL RESULT TABLE
# ============================================================

indicator_comparison = (
    pd.DataFrame(
        indicator_results
    )
)


# Best R² improvement first
indicator_comparison = (

    indicator_comparison
    .sort_values(
        "delta_r2",
        ascending=False
    )
    .reset_index(drop=True)

)


indicator_comparison


M1 Market | ETH Volume t+3
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-29 | n=729
Retrain: 2025-08-08 | Train through: 2025-08-05 | n=736
Retrain: 2025-08-15 | Train through: 2025-08-12 | n=743
Retrain: 2025-08-22 | Train through: 2025-08-19 | n=750
Retrain: 2025-08-29 | Train through: 2025-08-26 | n=757
Retrain: 2025-09-05 | Train through: 2025-09-02 | n=764
Retrain: 2025-09-12 | Train through: 2025-09-09 | n=771
Retrain: 2025-09-19 | Train through: 2025-09-16 | n=778
Retrain: 2025-09-26 | Train through: 2025-09-23 | n=785
Retrain: 2025-10-03 | Train through: 2025-09-30 | n=792
Retrain: 2025-10-10 | Train through: 2025-10-07 | n=799
Retrain: 2025-10-17 | Train through: 2025-10-14 | n=806
Retrain: 2025-10-24 | Train through: 2025-10-21 | n=813
Retrain: 2025-10-31 | Train through: 2025-10-28 | n=820
Retrain: 2025-11-07 | Train through: 2025-11-04 | n=827
Retrain: 2025-11-14 | Train through: 2025-11-11 | n=

,feature,n,mae,delta_mae,rmse,delta_rmse,r2,delta_r2,direction_accuracy,mean_loss_difference,dm_stat,dm_p_value
0,bridge_volume_momentum_7d,365,0.328752,-0.004307,0.415210,-0.003745,0.409183,0.010706,0.731507,0.003124,1.219899,0.222503
1,avg_transfer_size_usd,365,0.333347,0.000288,0.420536,0.001581,0.393928,-0.004548,0.717808,-0.001327,-0.745786,0.455797
2,optimism_volume_share,365,0.335346,0.002287,0.421968,0.003014,0.389792,-0.008685,0.723288,-0.002534,-2.174506,0.029667
3,arbitrum_volume_share,365,0.335720,0.002661,0.422400,0.003445,0.388543,-0.009933,0.715068,-0.002898,-2.501371,0.012371
4,net_flow_3d,365,0.335835,0.002776,0.422474,0.003519,0.388329,-0.010147,0.715068,-0.002961,-2.509833,0.012079
5,net_flow_30d,365,0.336266,0.003207,0.422551,0.003596,0.388106,-0.010370,0.723288,-0.003026,-2.474272,0.013351
6,base_volume_share,365,0.336121,0.003062,0.422567,0.003612,0.388058,-0.010418,0.712329,-0.003040,-2.564479,0.010333
7,tx_imbalance,365,0.335624,0.002565,0.422672,0.003717,0.387754,-0.010722,0.701370,-0.003129,-2.601776,0.009274
8,zksync_volume_share,365,0.335637,0.002578,0.422699,0.003745,0.387676,-0.010801,0.723288,-0.003152,-2.502370,0.012336
9,net_flow_7d,365,0.335515,0.002456,0.422867,0.003912,0.387190,-0.011287,0.715068,-0.003293,-2.545988,0.010897


In [66]:
# The table above is Table 3

In [67]:
import numpy as np
import pandas as pd
from scipy.stats import norm


# ============================================================
# 1. SETTINGS
# ============================================================

BRIDGE_FEATURE = "bridge_volume_momentum_7d"

LAGS = [0, 1, 2, 3, 7]

HORIZONS = [1, 3, 7]

# ============================================================
# 3. PREPARE DATE
# ============================================================

market_data = data_m1.copy()
bridge_data = data_m3.copy()


# ------------------------------------------------------------
# Market data
# ------------------------------------------------------------

if market_data.index.name == "Date":
    market_data = market_data.reset_index()

if "Date" not in market_data.columns:
    raise ValueError(
        "Could not find the Date column in data_m1."
    )

market_data["Date"] = pd.to_datetime(
    market_data["Date"]
)


# ------------------------------------------------------------
# Bridge data
# ------------------------------------------------------------

if bridge_data.index.name == "Date":
    bridge_data = bridge_data.reset_index()

if "Date" not in bridge_data.columns:
    raise ValueError(
        "Could not find the Date column in data_m3."
    )

bridge_data["Date"] = pd.to_datetime(
    bridge_data["Date"]
)


# ------------------------------------------------------------
# Check bridge feature
# ------------------------------------------------------------

if BRIDGE_FEATURE not in bridge_data.columns:

    raise ValueError(
        f"{BRIDGE_FEATURE} is missing from data_m3."
    )


# ============================================================
# 4. CREATE LAGGED BRIDGE DATA
# ============================================================

bridge_lag_data = (
    bridge_data[
        [
            "Date",
            BRIDGE_FEATURE
        ]
    ]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)


for lag in LAGS:

    if lag == 0:

        bridge_lag_data[
            f"{BRIDGE_FEATURE}_lag0"
        ] = bridge_lag_data[
            BRIDGE_FEATURE
        ]

    else:

        bridge_lag_data[
            f"{BRIDGE_FEATURE}_lag{lag}"
        ] = (
            bridge_lag_data[
                BRIDGE_FEATURE
            ]
            .shift(lag)
        )


print(
    bridge_lag_data[
        [
            "Date"
        ]
        +
        [
            f"{BRIDGE_FEATURE}_lag{x}"
            for x in LAGS
        ]
    ].head(10)
)


# ============================================================
# 5. STORAGE
# ============================================================

experiment_results = []


# ============================================================
# 6. RUN EACH HORIZON
# ============================================================

for horizon in HORIZONS:

    print()
    print("#" * 80)
    print(
        f"HORIZON t+{horizon}"
    )
    print("#" * 80)


    # ========================================================
    # 6.1 BASELINE M1
    # ========================================================

    baseline_results, _, baseline_summary = (
        test_volume(
            market_data,
            horizon=horizon,
            model_name=f"M1 Baseline t+{horizon}"
        )
    )


    # ========================================================
    # 6.2 EACH LAG
    # ========================================================

    for lag in LAGS:

        lag_col = (
            f"{BRIDGE_FEATURE}_lag{lag}"
        )


        print()
        print("=" * 70)
        print(
            f"Horizon={horizon} | Lag={lag}"
        )
        print("=" * 70)


        # ====================================================
        # Candidate = Market + ONE lagged bridge feature
        # ====================================================

        lag_temp = (
            bridge_lag_data[
                [
                    "Date",
                    lag_col
                ]
            ]
            .copy()
        )


        candidate_data = (
            market_data
            .merge(
                lag_temp,
                on="Date",
                how="inner"
            )
        )


        # ====================================================
        # Run forecasting
        # ====================================================

        candidate_results, _, candidate_summary = (
            test_volume(
                candidate_data,
                horizon=horizon,
                model_name=(
                    f"M1 + Momentum "
                    f"Lag {lag}"
                )
            )
        )


        # ====================================================
        # 6.3 ALIGN COMMON TEST DATES
        # ====================================================

        base_temp = (
            baseline_results[
                [
                    "feature_date",
                    "actual_volume_change",
                    "predicted_volume_change"
                ]
            ]
            .copy()
        )


        candidate_temp = (
            candidate_results[
                [
                    "feature_date",
                    "actual_volume_change",
                    "predicted_volume_change"
                ]
            ]
            .copy()
        )


        base_temp["feature_date"] = (
            pd.to_datetime(
                base_temp["feature_date"]
            )
        )


        candidate_temp["feature_date"] = (
            pd.to_datetime(
                candidate_temp["feature_date"]
            )
        )


        compare = (
            base_temp
            .merge(
                candidate_temp,
                on="feature_date",
                suffixes=(
                    "_baseline",
                    "_candidate"
                )
            )
            .sort_values(
                "feature_date"
            )
            .reset_index(drop=True)
        )


        # ====================================================
        # 6.4 CHECK TARGET
        # ====================================================

        same_actual = np.allclose(

            compare[
                "actual_volume_change_baseline"
            ],

            compare[
                "actual_volume_change_candidate"
            ]

        )


        if not same_actual:

            raise ValueError(
                f"Target mismatch: "
                f"horizon={horizon}, "
                f"lag={lag}"
            )


        # ====================================================
        # 6.5 ERRORS
        # ====================================================

        compare["error_baseline"] = (

            compare[
                "actual_volume_change_baseline"
            ]

            -

            compare[
                "predicted_volume_change_baseline"
            ]

        )


        compare["error_candidate"] = (

            compare[
                "actual_volume_change_candidate"
            ]

            -

            compare[
                "predicted_volume_change_candidate"
            ]

        )


        # ====================================================
        # 6.6 IMPORTANT:
        # Recalculate BOTH performances on common dates
        # ====================================================

        from sklearn.metrics import (
            mean_absolute_error,
            mean_squared_error,
            r2_score
        )


        y_true = compare[
            "actual_volume_change_baseline"
        ]


        y_base = compare[
            "predicted_volume_change_baseline"
        ]


        y_candidate = compare[
            "predicted_volume_change_candidate"
        ]


        # ----------------------------------------------------
        # Baseline
        # ----------------------------------------------------

        base_mae = mean_absolute_error(
            y_true,
            y_base
        )

        base_rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_base
            )
        )

        base_r2 = r2_score(
            y_true,
            y_base
        )


        # ----------------------------------------------------
        # Candidate
        # ----------------------------------------------------

        candidate_mae = mean_absolute_error(
            y_true,
            y_candidate
        )

        candidate_rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_candidate
            )
        )

        candidate_r2 = r2_score(
            y_true,
            y_candidate
        )


        # ====================================================
        # 6.7 DIRECTION ACCURACY
        # ====================================================

        actual_direction = (
            y_true > 0
        ).astype(int)


        base_direction = (
            y_base > 0
        ).astype(int)


        candidate_direction = (
            y_candidate > 0
        ).astype(int)


        base_direction_accuracy = (
            actual_direction.values
            ==
            base_direction.values
        ).mean()


        candidate_direction_accuracy = (
            actual_direction.values
            ==
            candidate_direction.values
        ).mean()


        # ====================================================
        # 6.8 DM TEST
        # ====================================================

        dm = dm_test_hac(

            compare["error_baseline"],

            compare["error_candidate"],

            horizon=horizon

        )


        # ====================================================
        # 6.9 STORE
        # ====================================================

        experiment_results.append({

            "horizon":
                horizon,

            "lag":
                lag,

            "n":
                len(compare),

            # Baseline
            "baseline_r2":
                base_r2,

            "baseline_rmse":
                base_rmse,

            "baseline_mae":
                base_mae,

            "baseline_direction_accuracy":
                base_direction_accuracy,

            # Candidate
            "candidate_r2":
                candidate_r2,

            "candidate_rmse":
                candidate_rmse,

            "candidate_mae":
                candidate_mae,

            "candidate_direction_accuracy":
                candidate_direction_accuracy,

            # Difference
            "delta_r2":
                candidate_r2
                -
                base_r2,

            "delta_rmse":
                candidate_rmse
                -
                base_rmse,

            "delta_mae":
                candidate_mae
                -
                base_mae,

            "delta_direction_accuracy":
                candidate_direction_accuracy
                -
                base_direction_accuracy,

            # DM
            "mean_loss_difference":
                dm[
                    "mean_loss_difference"
                ],

            "dm_stat":
                dm[
                    "dm_stat"
                ],

            "dm_p_value":
                dm[
                    "p_value"
                ]
        })


# ============================================================
# 7. FINAL DATAFRAME
# ============================================================

lag_experiment = pd.DataFrame(
    experiment_results
)


lag_experiment = (
    lag_experiment
    .sort_values(
        [
            "horizon",
            "lag"
        ]
    )
    .reset_index(drop=True)
)


lag_experiment

        Date  bridge_volume_momentum_7d_lag0  bridge_volume_momentum_7d_lag1  \
0 2023-03-14                       -0.170270                             NaN   
1 2023-03-15                        0.110870                       -0.170270   
2 2023-03-16                        0.483535                        0.110870   
3 2023-03-17                        0.004706                        0.483535   
4 2023-03-18                       -0.192975                        0.004706   
5 2023-03-19                        0.770941                       -0.192975   
6 2023-03-20                       -0.462317                        0.770941   
7 2023-03-21                       -0.243826                       -0.462317   
8 2023-03-22                        0.335189                       -0.243826   
9 2023-03-23                        2.079438                        0.335189   

   bridge_volume_momentum_7d_lag2  bridge_volume_momentum_7d_lag3  \
0                             NaN                 

,horizon,lag,n,baseline_r2,baseline_rmse,baseline_mae,baseline_direction_accuracy,candidate_r2,candidate_rmse,candidate_mae,candidate_direction_accuracy,delta_r2,delta_rmse,delta_mae,delta_direction_accuracy,mean_loss_difference,dm_stat,dm_p_value
0,1,0,365,0.248362,0.362972,0.278944,0.663014,0.254668,0.361446,0.276380,0.693151,0.006306,-0.001526,-0.002564,0.030137,0.001105,0.766662,0.443283
1,1,1,365,0.248362,0.362972,0.278944,0.663014,0.240354,0.364901,0.280932,0.663014,-0.008008,0.001929,0.001988,0.000000,-0.001404,-1.186754,0.235325
2,1,2,365,0.248362,0.362972,0.278944,0.663014,0.239076,0.365207,0.280863,0.668493,-0.009286,0.002235,0.001918,0.005479,-0.001628,-1.891250,0.058591
3,1,3,365,0.248362,0.362972,0.278944,0.663014,0.238607,0.365320,0.281403,0.671233,-0.009756,0.002348,0.002458,0.008219,-0.001710,-2.005508,0.044909
4,1,7,365,0.248362,0.362972,0.278944,0.663014,0.245299,0.363711,0.278024,0.673973,-0.003063,0.000739,-0.000920,0.010959,-0.000537,-0.436998,0.662113
5,3,0,365,0.398476,0.418955,0.333059,0.734247,0.409183,0.415210,0.328752,0.731507,0.010706,-0.003745,-0.004307,-0.002740,0.003124,1.219899,0.222503
6,3,1,365,0.398476,0.418955,0.333059,0.734247,0.387764,0.422669,0.335720,0.723288,-0.010712,0.003714,0.002661,-0.010959,-0.003126,-2.474907,0.013327
7,3,2,365,0.398476,0.418955,0.333059,0.734247,0.390140,0.421848,0.335277,0.715068,-0.008336,0.002893,0.002218,-0.019178,-0.002432,-2.020550,0.043326
8,3,3,365,0.398476,0.418955,0.333059,0.734247,0.396001,0.419816,0.333457,0.728767,-0.002475,0.000861,0.000398,-0.005479,-0.000722,-0.486179,0.626841
9,3,7,365,0.398476,0.418955,0.333059,0.734247,0.405344,0.416556,0.331898,0.715068,0.006868,-0.002399,-0.001161,-0.019178,0.002004,0.868343,0.385206


In [68]:
lag_experiment[
    [
        "horizon",
        "lag",
        "n",
        "baseline_r2",
        "candidate_r2",
        "delta_r2",
        "delta_rmse",
        "delta_mae",
        "delta_direction_accuracy",
        "dm_p_value"
    ]
]

,horizon,lag,n,baseline_r2,candidate_r2,delta_r2,delta_rmse,delta_mae,delta_direction_accuracy,dm_p_value
0,1,0,365,0.248362,0.254668,0.006306,-0.001526,-0.002564,0.030137,0.443283
1,1,1,365,0.248362,0.240354,-0.008008,0.001929,0.001988,0.000000,0.235325
2,1,2,365,0.248362,0.239076,-0.009286,0.002235,0.001918,0.005479,0.058591
3,1,3,365,0.248362,0.238607,-0.009756,0.002348,0.002458,0.008219,0.044909
4,1,7,365,0.248362,0.245299,-0.003063,0.000739,-0.000920,0.010959,0.662113
5,3,0,365,0.398476,0.409183,0.010706,-0.003745,-0.004307,-0.002740,0.222503
6,3,1,365,0.398476,0.387764,-0.010712,0.003714,0.002661,-0.010959,0.013327
7,3,2,365,0.398476,0.390140,-0.008336,0.002893,0.002218,-0.019178,0.043326
8,3,3,365,0.398476,0.396001,-0.002475,0.000861,0.000398,-0.005479,0.626841
9,3,7,365,0.398476,0.405344,0.006868,-0.002399,-0.001161,-0.019178,0.385206


In [69]:
lag_experiment.pivot(
    index="lag",
    columns="horizon",
    values="delta_r2"
)

horizon,1,3,7
lag,,,
0,0.006306,0.010706,-0.001767
1,-0.008008,-0.010712,-0.000805
2,-0.009286,-0.008336,-0.002415
3,-0.009756,-0.002475,-0.001909
7,-0.003063,0.006868,-0.000913


In [70]:
lag_experiment.pivot(
    index="lag",
    columns="horizon",
    values="dm_p_value"
)

horizon,1,3,7
lag,,,
0,0.443283,0.222503,0.740640
1,0.235325,0.013327,0.892503
2,0.058591,0.043326,0.695089
3,0.044909,0.626841,0.746520
7,0.662113,0.385206,0.870096


In [72]:
# momentum... 

import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

WINDOWS = [3, 7, 14, 30]
HORIZONS = [1, 3, 7]

BRIDGE_VOLUME_COL = "gross_volume_usd"


# ============================================================
# 2. PREPARE BRIDGE DATA
# ============================================================

bridge_source = data_m4.copy()

if bridge_source.index.name == "Date":
    bridge_source = bridge_source.reset_index()

bridge_source["Date"] = pd.to_datetime(
    bridge_source["Date"]
)

bridge_source = (
    bridge_source
    .sort_values("Date")
    .reset_index(drop=True)
)


# ============================================================
# 3. CREATE BRIDGE VOLUME MOMENTUM
# ============================================================
#
# momentum_k:
#
# current bridge volume / k-day moving average - 1
#
# Example:
#
# momentum_7d = 0.50
#
# Current bridge volume is approximately
# 50% above its trailing 7-day average
#
# ============================================================

momentum_cols = []

for window in WINDOWS:

    col = f"bridge_volume_momentum_{window}d"

    bridge_source[col] = (
        bridge_source[BRIDGE_VOLUME_COL]
        /
        bridge_source[
            BRIDGE_VOLUME_COL
        ].rolling(
            window=window,
            min_periods=window
        ).mean()
        -
        1
    )

    momentum_cols.append(col)


print(
    bridge_source[
        ["Date"] + momentum_cols
    ].tail()
)

# ============================================================
# 4. PREPARE MARKET BASELINE
# ============================================================

market = data_m1.copy()

if market.index.name == "Date":
    market = market.reset_index()

market["Date"] = pd.to_datetime(
    market["Date"]
)


# ============================================================
# 5. RUN WINDOW × HORIZON EXPERIMENT
# ============================================================

window_results = []


for horizon in HORIZONS:

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline_results, _, baseline_summary = (
        test_volume(
            market,
            horizon=horizon,
            model_name=f"M1 t+{horizon}"
        )
    )


    for window in WINDOWS:

        feature = (
            f"bridge_volume_momentum_{window}d"
        )


        print()
        print("=" * 70)
        print(
            f"Window={window} | "
            f"Horizon={horizon}"
        )
        print("=" * 70)


        # ====================================================
        # Market + one momentum feature
        # ====================================================

        bridge_temp = (
            bridge_source[
                [
                    "Date",
                    feature
                ]
            ]
            .copy()
        )


        candidate_data = (
            market
            .merge(
                bridge_temp,
                on="Date",
                how="inner"
            )
        )


        candidate_results, _, candidate_summary = (
            test_volume(
                candidate_data,
                horizon=horizon,
                model_name=(
                    f"M1 + Momentum {window}d"
                )
            )
        )


        # ====================================================
        # 6. COMMON DATES
        # ====================================================

        baseline_temp = (
            baseline_results[
                [
                    "feature_date",
                    "actual_volume_change",
                    "predicted_volume_change"
                ]
            ]
            .copy()
        )


        candidate_temp = (
            candidate_results[
                [
                    "feature_date",
                    "actual_volume_change",
                    "predicted_volume_change"
                ]
            ]
            .copy()
        )


        baseline_temp["feature_date"] = (
            pd.to_datetime(
                baseline_temp["feature_date"]
            )
        )


        candidate_temp["feature_date"] = (
            pd.to_datetime(
                candidate_temp["feature_date"]
            )
        )


        compare = (
            baseline_temp
            .merge(
                candidate_temp,
                on="feature_date",
                suffixes=(
                    "_baseline",
                    "_candidate"
                )
            )
            .sort_values(
                "feature_date"
            )
            .reset_index(drop=True)
        )


        # ====================================================
        # 7. CHECK TARGET
        # ====================================================

        if not np.allclose(
            compare[
                "actual_volume_change_baseline"
            ],
            compare[
                "actual_volume_change_candidate"
            ]
        ):

            raise ValueError(
                f"Target mismatch: "
                f"window={window}, "
                f"horizon={horizon}"
            )


        # ====================================================
        # 8. RE-CALCULATE METRICS ON COMMON DATES
        # ====================================================

        from sklearn.metrics import (
            mean_absolute_error,
            mean_squared_error,
            r2_score
        )


        y_true = (
            compare[
                "actual_volume_change_baseline"
            ]
        )

        y_base = (
            compare[
                "predicted_volume_change_baseline"
            ]
        )

        y_candidate = (
            compare[
                "predicted_volume_change_candidate"
            ]
        )


        # Baseline metrics
        baseline_mae = mean_absolute_error(
            y_true,
            y_base
        )

        baseline_rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_base
            )
        )

        baseline_r2 = r2_score(
            y_true,
            y_base
        )


        # Candidate metrics
        candidate_mae = mean_absolute_error(
            y_true,
            y_candidate
        )

        candidate_rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_candidate
            )
        )

        candidate_r2 = r2_score(
            y_true,
            y_candidate
        )


        # ====================================================
        # 9. DIRECTION ACCURACY
        # ====================================================

        true_direction = (
            y_true > 0
        ).astype(int)

        base_direction = (
            y_base > 0
        ).astype(int)

        candidate_direction = (
            y_candidate > 0
        ).astype(int)


        baseline_direction_accuracy = (
            true_direction.values
            ==
            base_direction.values
        ).mean()


        candidate_direction_accuracy = (
            true_direction.values
            ==
            candidate_direction.values
        ).mean()


        # ====================================================
        # 10. ERRORS + DM TEST
        # ====================================================

        error_base = (
            y_true
            -
            y_base
        )

        error_candidate = (
            y_true
            -
            y_candidate
        )


        dm = dm_test_hac(
            error_base,
            error_candidate,
            horizon=horizon
        )


        # ====================================================
        # 11. SAVE
        # ====================================================

        window_results.append({

            "window":
                window,

            "horizon":
                horizon,

            "n":
                len(compare),

            "baseline_r2":
                baseline_r2,

            "candidate_r2":
                candidate_r2,

            "delta_r2":
                candidate_r2
                -
                baseline_r2,

            "baseline_rmse":
                baseline_rmse,

            "candidate_rmse":
                candidate_rmse,

            "delta_rmse":
                candidate_rmse
                -
                baseline_rmse,

            "baseline_mae":
                baseline_mae,

            "candidate_mae":
                candidate_mae,

            "delta_mae":
                candidate_mae
                -
                baseline_mae,

            "baseline_direction_accuracy":
                baseline_direction_accuracy,

            "candidate_direction_accuracy":
                candidate_direction_accuracy,

            "delta_direction_accuracy":
                candidate_direction_accuracy
                -
                baseline_direction_accuracy,

            "dm_stat":
                dm["dm_stat"],

            "dm_p_value":
                dm["p_value"]

        })



           Date  bridge_volume_momentum_3d  bridge_volume_momentum_7d  \
1235 2026-07-31                  -0.794694                  -0.647667   
1236 2026-08-01                  -0.317944                  -0.699987   
1237 2026-08-02                   0.003532                  -0.682750   
1238 2026-08-03                   0.338941                  -0.518619   
1239 2026-08-04                  -0.987798                  -0.996180   

      bridge_volume_momentum_14d  bridge_volume_momentum_30d  
1235                   -0.521019                   -0.531899  
1236                   -0.580950                   -0.591384  
1237                   -0.545427                   -0.543835  
1238                   -0.282397                   -0.295952  
1239                   -0.994915                   -0.995193  

M1 t+1 | ETH Volume t+1
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-31 | n=731
Retrain: 2025-08-08 | 

In [73]:
# ============================================================
# 12. RESULT TABLE
# ============================================================

momentum_window_results = (
    pd.DataFrame(
        window_results
    )
    .sort_values(
        [
            "horizon",
            "window"
        ]
    )
    .reset_index(drop=True)
)


momentum_window_results[
    [
        "window",
        "horizon",
        "n",
        "baseline_r2",
        "candidate_r2",
        "delta_r2",
        "delta_rmse",
        "delta_mae",
        "delta_direction_accuracy",
        "dm_p_value"
    ]
]

,window,horizon,n,baseline_r2,candidate_r2,delta_r2,delta_rmse,delta_mae,delta_direction_accuracy,dm_p_value
0,3,1,365,0.248362,0.243490,-0.004872,0.001175,0.000656,0.013699,0.350216
1,7,1,365,0.248362,0.254668,0.006306,-0.001526,-0.002564,0.030137,0.443283
2,14,1,365,0.248362,0.248480,0.000118,-0.000029,-0.001211,0.008219,0.987839
3,30,1,365,0.248362,0.244113,-0.004250,0.001025,-0.000306,0.010959,0.582600
4,3,3,365,0.398476,0.393793,-0.004683,0.001628,-0.000252,0.002740,0.385095
5,7,3,365,0.398476,0.409183,0.010706,-0.003745,-0.004307,-0.002740,0.222503
6,14,3,365,0.398476,0.396458,-0.002018,0.000702,-0.000592,-0.013699,0.785531
7,30,3,365,0.398476,0.399452,0.000975,-0.000340,-0.000554,-0.002740,0.880116
8,3,7,362,0.218784,0.214960,-0.003824,0.000889,-0.001351,0.011050,0.511565
9,7,7,362,0.218784,0.217017,-0.001767,0.000411,-0.001608,0.008287,0.740640


In [74]:
momentum_window_results.pivot(
    index="window",
    columns="horizon",
    values="delta_r2"
)

horizon,1,3,7
window,,,
3,-0.004872,-0.004683,-0.003824
7,0.006306,0.010706,-0.001767
14,0.000118,-0.002018,-0.000108
30,-0.004250,0.000975,-0.000790


In [75]:
# ============================================================
# REGIME ANALYSIS
# ============================================================

regime_df = data_m4.copy()

if regime_df.index.name == "Date":
    regime_df = regime_df.reset_index()

regime_df["Date"] = pd.to_datetime(regime_df["Date"])

feature = "bridge_volume_momentum_7d"


# ------------------------------------------------------------
# High / Normal regime threshold
# ------------------------------------------------------------

threshold = (
    regime_df[feature]
    .quantile(0.75)
)


regime_df["bridge_regime"] = np.where(
    regime_df[feature] >= threshold,
    "High",
    "Normal"
)


print(
    regime_df[
        "bridge_regime"
    ].value_counts()
)

bridge_regime
Normal    930
High      310
Name: count, dtype: int64


In [76]:
# ============================================================
# MERGE REGIME WITH RESULTS
# ============================================================

regime_info = (
    regime_df[
        [
            "Date",
            "bridge_regime"
        ]
    ]
    .rename(
        columns={
            "Date":
                "feature_date"
        }
    )
)


m1 = (
    results_h3_m1
    .merge(
        regime_info,
        on="feature_date",
        how="inner"
    )
)


m3 = (
    results_h3_m3
    .merge(
        regime_info,
        on="feature_date",
        how="inner"
    )
)

In [77]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


regime_results = []


for regime in [
    "Normal",
    "High"
]:

    m1_r = m1[
        m1["bridge_regime"]
        ==
        regime
    ]

    m3_r = m3[
        m3["bridge_regime"]
        ==
        regime
    ]


    common_dates = sorted(
        set(
            m1_r["feature_date"]
        )
        &
        set(
            m3_r["feature_date"]
        )
    )


    m1_r = (
        m1_r[
            m1_r["feature_date"]
            .isin(common_dates)
        ]
        .sort_values(
            "feature_date"
        )
    )


    m3_r = (
        m3_r[
            m3_r["feature_date"]
            .isin(common_dates)
        ]
        .sort_values(
            "feature_date"
        )
    )


    y_true = (
        m1_r[
            "actual_volume_change"
        ]
    )


    y_m1 = (
        m1_r[
            "predicted_volume_change"
        ]
    )


    y_m3 = (
        m3_r[
            "predicted_volume_change"
        ]
    )


    # Baseline
    r2_m1 = r2_score(
        y_true,
        y_m1
    )

    rmse_m1 = np.sqrt(
        mean_squared_error(
            y_true,
            y_m1
        )
    )


    # Bridge
    r2_m3 = r2_score(
        y_true,
        y_m3
    )

    rmse_m3 = np.sqrt(
        mean_squared_error(
            y_true,
            y_m3
        )
    )


    regime_results.append({

        "regime":
            regime,

        "n":
            len(y_true),

        "r2_m1":
            r2_m1,

        "r2_m3":
            r2_m3,

        "delta_r2":
            r2_m3
            -
            r2_m1,

        "rmse_m1":
            rmse_m1,

        "rmse_m3":
            rmse_m3,

        "delta_rmse":
            rmse_m3
            -
            rmse_m1

    })


regime_results = pd.DataFrame(
    regime_results
)

regime_results

,regime,n,r2_m1,r2_m3,delta_r2,rmse_m1,rmse_m3,delta_rmse
0,Normal,280,0.429342,0.433703,0.004362,0.408061,0.406499,-0.001562
1,High,85,0.090634,0.117576,0.026942,0.452990,0.446229,-0.006761


In [78]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import norm


# ============================================================
# 1. SETTINGS
# ============================================================

FEATURE = "bridge_volume_momentum_7d"

HORIZON = 3

THRESHOLDS = [
    0.70,
    0.75,
    0.80,
    0.90
]


# ============================================================
# 2. DM TEST WITH HAC
# ============================================================

def dm_test_hac(
    error1,
    error2,
    horizon=3
):

    e1 = np.asarray(
        error1,
        dtype=float
    )

    e2 = np.asarray(
        error2,
        dtype=float
    )


    # --------------------------------------------------------
    # Squared error loss differential
    #
    # d > 0:
    # baseline M1 has larger error
    # -> M3 better
    # --------------------------------------------------------

    d = (
        e1 ** 2
        -
        e2 ** 2
    )


    d = d[
        np.isfinite(d)
    ]


    T = len(d)


    if T < 5:

        return {
            "mean_loss_difference":
                np.nan,

            "dm_stat":
                np.nan,

            "p_value":
                np.nan
        }


    mean_d = (
        np.mean(d)
    )


    centered = (
        d
        -
        mean_d
    )


    # --------------------------------------------------------
    # HAC / Newey-West long-run variance
    # --------------------------------------------------------

    gamma0 = (
        np.sum(
            centered
            *
            centered
        )
        /
        T
    )


    long_run_variance = (
        gamma0
    )


    max_lag = max(
        horizon - 1,
        0
    )


    for lag in range(
        1,
        max_lag + 1
    ):

        gamma = (

            np.sum(
                centered[lag:]
                *
                centered[:-lag]
            )

            /

            T
        )


        weight = (

            1

            -

            lag
            /
            (max_lag + 1)

        )


        long_run_variance += (

            2

            *

            weight

            *

            gamma

        )


    if long_run_variance <= 0:

        return {
            "mean_loss_difference":
                mean_d,

            "dm_stat":
                np.nan,

            "p_value":
                np.nan
        }


    variance_mean = (

        long_run_variance

        /

        T

    )


    dm_stat = (

        mean_d

        /

        np.sqrt(
            variance_mean
        )

    )


    p_value = (

        2

        *

        (
            1
            -
            norm.cdf(
                abs(dm_stat)
            )
        )

    )


    return {

        "mean_loss_difference":
            mean_d,

        "dm_stat":
            dm_stat,

        "p_value":
            p_value
    }


# ============================================================
# 3. PREPARE BRIDGE REGIME DATA
# ============================================================

regime_source = data_m4.copy()


if regime_source.index.name == "Date":

    regime_source = (
        regime_source
        .reset_index()
    )


regime_source["Date"] = pd.to_datetime(
    regime_source["Date"]
)


regime_source = (
    regime_source
    .sort_values("Date")
    .reset_index(drop=True)
)


# Check feature
if FEATURE not in regime_source.columns:

    raise ValueError(
        f"{FEATURE} not found."
    )


regime_source[FEATURE] = pd.to_numeric(
    regime_source[FEATURE],
    errors="coerce"
)


# ============================================================
# 4. PREPARE FORECAST RESULTS
# ============================================================

m1_base = results_h3_m1.copy()
m3_base = results_h3_m3.copy()


m1_base["feature_date"] = pd.to_datetime(
    m1_base["feature_date"]
)


m3_base["feature_date"] = pd.to_datetime(
    m3_base["feature_date"]
)


# ============================================================
# 5. FUNCTION:
# EXPANDING REGIME ANALYSIS
# ============================================================

def run_regime_analysis(
    quantile_level
):

    temp = (
        regime_source[
            [
                "Date",
                FEATURE
            ]
        ]
        .copy()
    )


    # ========================================================
    # Expanding quantile
    #
    # IMPORTANT:
    #
    # today's threshold uses ONLY previous observations
    #
    # shift(1) prevents today's bridge value from
    # participating in its own threshold calculation
    # ========================================================

    temp["expanding_threshold"] = (

        temp[FEATURE]
        .shift(1)
        .expanding(
            min_periods=30
        )
        .quantile(
            quantile_level
        )

    )


    # ========================================================
    # Define regime
    # ========================================================

    temp["bridge_regime"] = np.where(

        temp[FEATURE]
        >=
        temp["expanding_threshold"],

        "High",

        "Normal"

    )


    # rows without enough historical observations
    temp.loc[
        temp["expanding_threshold"].isna(),
        "bridge_regime"
    ] = np.nan


    # --------------------------------------------------------
    # Rename date
    # --------------------------------------------------------

    regime_info = (

        temp[
            [
                "Date",
                FEATURE,
                "expanding_threshold",
                "bridge_regime"
            ]
        ]

        .rename(
            columns={
                "Date":
                    "feature_date"
            }
        )

    )


    # ========================================================
    # Merge regime into M1 / M3 results
    # ========================================================

    m1 = (

        m1_base

        .merge(
            regime_info,
            on="feature_date",
            how="inner"
        )

    )


    m3 = (

        m3_base

        .merge(
            regime_info,
            on="feature_date",
            how="inner"
        )

    )


    # drop undefined regime rows
    m1 = m1.dropna(
        subset=["bridge_regime"]
    )

    m3 = m3.dropna(
        subset=["bridge_regime"]
    )


    # ========================================================
    # 6. REGIME-BY-REGIME ANALYSIS
    # ========================================================

    rows = []


    for regime in [
        "Normal",
        "High"
    ]:


        m1_r = (

            m1[
                m1["bridge_regime"]
                ==
                regime
            ]

            .copy()

        )


        m3_r = (

            m3[
                m3["bridge_regime"]
                ==
                regime
            ]

            .copy()

        )


        # ====================================================
        # Common dates
        # ====================================================

        common_dates = sorted(

            set(
                m1_r["feature_date"]
            )

            &

            set(
                m3_r["feature_date"]
            )

        )


        m1_r = (

            m1_r[
                m1_r[
                    "feature_date"
                ].isin(
                    common_dates
                )
            ]

            .sort_values(
                "feature_date"
            )

            .reset_index(drop=True)

        )


        m3_r = (

            m3_r[
                m3_r[
                    "feature_date"
                ].isin(
                    common_dates
                )
            ]

            .sort_values(
                "feature_date"
            )

            .reset_index(drop=True)

        )


        if len(m1_r) < 10:

            continue


        # ====================================================
        # Verify actual target
        # ====================================================

        same_actual = np.allclose(

            m1_r[
                "actual_volume_change"
            ],

            m3_r[
                "actual_volume_change"
            ]

        )


        if not same_actual:

            raise ValueError(
                f"Target mismatch "
                f"for regime={regime}, "
                f"q={quantile_level}"
            )


        # ====================================================
        # Actual / predictions
        # ====================================================

        y_true = (

            m1_r[
                "actual_volume_change"
            ]

        )


        y_m1 = (

            m1_r[
                "predicted_volume_change"
            ]

        )


        y_m3 = (

            m3_r[
                "predicted_volume_change"
            ]

        )


        # ====================================================
        # Metrics
        # ====================================================

        mae_m1 = (
            mean_absolute_error(
                y_true,
                y_m1
            )
        )


        mae_m3 = (
            mean_absolute_error(
                y_true,
                y_m3
            )
        )


        rmse_m1 = np.sqrt(

            mean_squared_error(
                y_true,
                y_m1
            )

        )


        rmse_m3 = np.sqrt(

            mean_squared_error(
                y_true,
                y_m3
            )

        )


        r2_m1 = (
            r2_score(
                y_true,
                y_m1
            )
        )


        r2_m3 = (
            r2_score(
                y_true,
                y_m3
            )
        )


        # ====================================================
        # Direction accuracy
        # ====================================================

        actual_dir = (
            y_true > 0
        ).astype(int)


        dir_m1 = (
            y_m1 > 0
        ).astype(int)


        dir_m3 = (
            y_m3 > 0
        ).astype(int)


        direction_m1 = (

            actual_dir.values
            ==
            dir_m1.values

        ).mean()


        direction_m3 = (

            actual_dir.values
            ==
            dir_m3.values

        ).mean()


        # ====================================================
        # Forecast errors
        # ====================================================

        error_m1 = (
            y_true
            -
            y_m1
        )


        error_m3 = (
            y_true
            -
            y_m3
        )


        # ====================================================
        # DM TEST
        # ====================================================

        dm = dm_test_hac(
            error_m1,
            error_m3,
            horizon=HORIZON
        )


        # ====================================================
        # SAVE
        # ====================================================

        rows.append({

            "quantile":
                quantile_level,

            "regime":
                regime,

            "n":
                len(y_true),

            "r2_m1":
                r2_m1,

            "r2_m3":
                r2_m3,

            "delta_r2":
                r2_m3
                -
                r2_m1,

            "rmse_m1":
                rmse_m1,

            "rmse_m3":
                rmse_m3,

            "delta_rmse":
                rmse_m3
                -
                rmse_m1,

            "mae_m1":
                mae_m1,

            "mae_m3":
                mae_m3,

            "delta_mae":
                mae_m3
                -
                mae_m1,

            "direction_m1":
                direction_m1,

            "direction_m3":
                direction_m3,

            "delta_direction":
                direction_m3
                -
                direction_m1,

            "mean_loss_difference":
                dm[
                    "mean_loss_difference"
                ],

            "dm_stat":
                dm[
                    "dm_stat"
                ],

            "dm_p_value":
                dm[
                    "p_value"
                ]

        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 7. RUN THRESHOLD ROBUSTNESS
# ============================================================

all_regime_results = []


for q in THRESHOLDS:

    print(
        f"Running quantile = {q}"
    )


    temp_result = (
        run_regime_analysis(
            q
        )
    )


    all_regime_results.append(
        temp_result
    )


regime_robustness = (

    pd.concat(
        all_regime_results,
        ignore_index=True
    )

)


# ============================================================
# 8. FINAL TABLE
# ============================================================

regime_robustness[
    [
        "quantile",
        "regime",
        "n",

        "r2_m1",
        "r2_m3",
        "delta_r2",

        "delta_rmse",
        "delta_mae",

        "delta_direction",

        "dm_stat",
        "dm_p_value"
    ]
]

Running quantile = 0.7
Running quantile = 0.75
Running quantile = 0.8
Running quantile = 0.9


,quantile,regime,n,r2_m1,r2_m3,delta_r2,delta_rmse,delta_mae,delta_direction,dm_stat,dm_p_value
0,0.70,Normal,263,0.451779,0.450120,-0.001660,0.000599,-0.002240,-0.007605,-0.135999,0.891822
1,0.70,High,102,0.026953,0.069461,0.042508,-0.010452,-0.007339,0.019608,1.888874,0.058909
2,0.75,Normal,280,0.429342,0.433703,0.004362,-0.001562,-0.003977,-0.007143,0.384203,0.700828
3,0.75,High,85,0.090634,0.117576,0.026942,-0.006761,-0.002636,0.023529,1.128162,0.259252
4,0.80,Normal,298,0.422842,0.427992,0.005150,-0.001832,-0.004221,-0.006711,0.474061,0.635456
5,0.80,High,67,0.095229,0.122729,0.027500,-0.007010,-0.001194,0.029851,0.933780,0.350417
6,0.90,Normal,332,0.398223,0.408069,0.009846,-0.003402,-0.004560,0.000000,0.937456,0.348524
7,0.90,High,33,0.151622,0.144608,-0.007014,0.001915,0.005340,0.000000,-0.175501,0.860686


In [79]:
high_regime_results = (
    regime_robustness[
        regime_robustness["regime"]
        ==
        "High"
    ]
    .copy()
)

high_regime_results[
    [
        "quantile",
        "n",
        "r2_m1",
        "r2_m3",
        "delta_r2",
        "delta_rmse",
        "delta_mae",
        "dm_p_value"
    ]
]

,quantile,n,r2_m1,r2_m3,delta_r2,delta_rmse,delta_mae,dm_p_value
1,0.70,102,0.026953,0.069461,0.042508,-0.010452,-0.007339,0.058909
3,0.75,85,0.090634,0.117576,0.026942,-0.006761,-0.002636,0.259252
5,0.80,67,0.095229,0.122729,0.027500,-0.007010,-0.001194,0.350417
7,0.90,33,0.151622,0.144608,-0.007014,0.001915,0.005340,0.860686


In [80]:
high_regime_results[
    [
        "quantile",
        "delta_r2"
    ]
]

,quantile,delta_r2
1,0.70,0.042508
3,0.75,0.026942
5,0.80,0.027500
7,0.90,-0.007014


In [81]:
high_regime_results[
    [
        "quantile",
        "dm_p_value"
    ]
]

,quantile,dm_p_value
1,0.70,0.058909
3,0.75,0.259252
5,0.80,0.350417
7,0.90,0.860686


In [82]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. SETTINGS
# ============================================================

FEATURE = "bridge_volume_momentum_7d"
HORIZON = 3

LOW_Q  = 0.70
HIGH_Q = 0.90


# ============================================================
# 2. PREPARE REGIME SOURCE
# ============================================================

regime_source = data_m4.copy()

if regime_source.index.name == "Date":
    regime_source = regime_source.reset_index()

regime_source["Date"] = pd.to_datetime(
    regime_source["Date"]
)

regime_source = (
    regime_source
    .sort_values("Date")
    .reset_index(drop=True)
)


if FEATURE not in regime_source.columns:
    raise ValueError(
        f"{FEATURE} not found in data_m4"
    )


regime_source[FEATURE] = pd.to_numeric(
    regime_source[FEATURE],
    errors="coerce"
)


# ============================================================
# 3. EXPANDING QUANTILE THRESHOLDS
# ============================================================
#
# IMPORTANT:
# shift(1) ensures today's regime threshold
# uses only information available before today.
# ============================================================

regime_source["q70"] = (
    regime_source[FEATURE]
    .shift(1)
    .expanding(
        min_periods=30
    )
    .quantile(
        LOW_Q
    )
)


regime_source["q90"] = (
    regime_source[FEATURE]
    .shift(1)
    .expanding(
        min_periods=30
    )
    .quantile(
        HIGH_Q
    )
)


# ============================================================
# 4. DEFINE THREE REGIMES
# ============================================================

def classify_regime(row):

    if pd.isna(row["q70"]) or pd.isna(row["q90"]):
        return np.nan

    x = row[FEATURE]

    if x < row["q70"]:
        return "Normal"

    elif x < row["q90"]:
        return "High"

    else:
        return "Extreme"


regime_source["bridge_regime"] = (
    regime_source.apply(
        classify_regime,
        axis=1
    )
)


print(
    regime_source[
        "bridge_regime"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# 5. REGIME INFO
# ============================================================

regime_info = (
    regime_source[
        [
            "Date",
            FEATURE,
            "q70",
            "q90",
            "bridge_regime"
        ]
    ]
    .rename(
        columns={
            "Date":
                "feature_date"
        }
    )
)


# ============================================================
# 6. PREPARE M1 / M3 RESULTS
# ============================================================

m1 = results_h3_m1.copy()
m3 = results_h3_m3.copy()


m1["feature_date"] = pd.to_datetime(
    m1["feature_date"]
)

m3["feature_date"] = pd.to_datetime(
    m3["feature_date"]
)


m1 = (
    m1
    .merge(
        regime_info,
        on="feature_date",
        how="inner"
    )
)


m3 = (
    m3
    .merge(
        regime_info,
        on="feature_date",
        how="inner"
    )
)


m1 = m1.dropna(
    subset=["bridge_regime"]
)

m3 = m3.dropna(
    subset=["bridge_regime"]
)


# ============================================================
# 7. THREE-REGIME ANALYSIS
# ============================================================

three_regime_results = []


for regime in [
    "Normal",
    "High",
    "Extreme"
]:

    m1_r = (
        m1[
            m1["bridge_regime"]
            ==
            regime
        ]
        .copy()
    )

    m3_r = (
        m3[
            m3["bridge_regime"]
            ==
            regime
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Common dates
    # --------------------------------------------------------

    common_dates = sorted(
        set(
            m1_r["feature_date"]
        )
        &
        set(
            m3_r["feature_date"]
        )
    )


    m1_r = (
        m1_r[
            m1_r["feature_date"]
            .isin(
                common_dates
            )
        ]
        .sort_values(
            "feature_date"
        )
        .reset_index(drop=True)
    )


    m3_r = (
        m3_r[
            m3_r["feature_date"]
            .isin(
                common_dates
            )
        ]
        .sort_values(
            "feature_date"
        )
        .reset_index(drop=True)
    )


    if len(m1_r) < 10:

        print(
            f"Skip {regime}: "
            f"only {len(m1_r)} observations"
        )

        continue


    # ========================================================
    # 8. CHECK SAME TARGET
    # ========================================================

    if not np.allclose(
        m1_r[
            "actual_volume_change"
        ],
        m3_r[
            "actual_volume_change"
        ]
    ):

        raise ValueError(
            f"Target mismatch in {regime}"
        )


    # ========================================================
    # 9. VALUES
    # ========================================================

    y_true = (
        m1_r[
            "actual_volume_change"
        ]
    )

    y_m1 = (
        m1_r[
            "predicted_volume_change"
        ]
    )

    y_m3 = (
        m3_r[
            "predicted_volume_change"
        ]
    )


    # ========================================================
    # 10. METRICS
    # ========================================================

    mae_m1 = mean_absolute_error(
        y_true,
        y_m1
    )

    mae_m3 = mean_absolute_error(
        y_true,
        y_m3
    )


    rmse_m1 = np.sqrt(
        mean_squared_error(
            y_true,
            y_m1
        )
    )

    rmse_m3 = np.sqrt(
        mean_squared_error(
            y_true,
            y_m3
        )
    )


    r2_m1 = r2_score(
        y_true,
        y_m1
    )

    r2_m3 = r2_score(
        y_true,
        y_m3
    )


    # ========================================================
    # 11. DIRECTION ACCURACY
    # ========================================================

    actual_dir = (
        y_true > 0
    ).astype(int)

    m1_dir = (
        y_m1 > 0
    ).astype(int)

    m3_dir = (
        y_m3 > 0
    ).astype(int)


    direction_m1 = (
        actual_dir.values
        ==
        m1_dir.values
    ).mean()


    direction_m3 = (
        actual_dir.values
        ==
        m3_dir.values
    ).mean()


    # ========================================================
    # 12. DM TEST
    # ========================================================

    error_m1 = (
        y_true
        -
        y_m1
    )

    error_m3 = (
        y_true
        -
        y_m3
    )


    dm = dm_test_hac(
        error_m1,
        error_m3,
        horizon=HORIZON
    )


    # ========================================================
    # 13. SAVE RESULT
    # ========================================================

    three_regime_results.append({

        "regime":
            regime,

        "n":
            len(y_true),

        "r2_m1":
            r2_m1,

        "r2_m3":
            r2_m3,

        "delta_r2":
            r2_m3
            -
            r2_m1,

        "rmse_m1":
            rmse_m1,

        "rmse_m3":
            rmse_m3,

        "delta_rmse":
            rmse_m3
            -
            rmse_m1,

        "mae_m1":
            mae_m1,

        "mae_m3":
            mae_m3,

        "delta_mae":
            mae_m3
            -
            mae_m1,

        "direction_m1":
            direction_m1,

        "direction_m3":
            direction_m3,

        "delta_direction":
            direction_m3
            -
            direction_m1,

        "mean_loss_difference":
            dm[
                "mean_loss_difference"
            ],

        "dm_stat":
            dm[
                "dm_stat"
            ],

        "dm_p_value":
            dm[
                "p_value"
            ]
    })


# ============================================================
# 14. FINAL TABLE
# ============================================================

three_regime_results = pd.DataFrame(
    three_regime_results
)


three_regime_results[
    [
        "regime",
        "n",

        "r2_m1",
        "r2_m3",
        "delta_r2",

        "delta_rmse",
        "delta_mae",

        "direction_m1",
        "direction_m3",
        "delta_direction",

        "dm_stat",
        "dm_p_value"
    ]
]

bridge_regime
Normal     856
High       249
Extreme    105
NaN         30
Name: count, dtype: int64


,regime,n,r2_m1,r2_m3,delta_r2,delta_rmse,delta_mae,direction_m1,direction_m3,delta_direction,dm_stat,dm_p_value
0,Normal,263,0.451779,0.450120,-0.001660,0.000599,-0.002240,0.745247,0.737643,-0.007605,-0.135999,0.891822
1,High,69,-0.061265,0.010018,0.071283,-0.016315,-0.013403,0.666667,0.695652,0.028986,2.721558,0.006497
2,Extreme,33,0.151622,0.144608,-0.007014,0.001915,0.005340,0.787879,0.787879,0.000000,-0.175501,0.860686


In [83]:
three_regime_results[
    [
        "regime",
        "n",
        "delta_r2",
        "delta_rmse",
        "dm_p_value"
    ]
]

,regime,n,delta_r2,delta_rmse,dm_p_value
0,Normal,263,-0.001660,0.000599,0.891822
1,High,69,0.071283,-0.016315,0.006497
2,Extreme,33,-0.007014,0.001915,0.860686


## 4. Bridge activity quantile analysis

In [84]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. SETTINGS
# ============================================================

FEATURE = "bridge_volume_momentum_7d"
HORIZON = 3

MIN_HISTORY = 30


# ============================================================
# 2. PREPARE BRIDGE DATA
# ============================================================

regime_source = data_m4.copy()


if regime_source.index.name == "Date":
    regime_source = regime_source.reset_index()


regime_source["Date"] = pd.to_datetime(
    regime_source["Date"]
)


regime_source = (
    regime_source
    .sort_values("Date")
    .reset_index(drop=True)
)


regime_source[FEATURE] = pd.to_numeric(
    regime_source[FEATURE],
    errors="coerce"
)


# ============================================================
# 3. EXPANDING QUINTILE THRESHOLDS
# ============================================================
#
# For each date t, thresholds use
#
# only observations before t
#
# q20 = 20%
# q40 = 40%
# q60 = 60%
# q80 = 80%
#
# ============================================================

past_feature = (
    regime_source[FEATURE]
    .shift(1)
)


regime_source["q20"] = (
    past_feature
    .expanding(
        min_periods=MIN_HISTORY
    )
    .quantile(0.20)
)


regime_source["q40"] = (
    past_feature
    .expanding(
        min_periods=MIN_HISTORY
    )
    .quantile(0.40)
)


regime_source["q60"] = (
    past_feature
    .expanding(
        min_periods=MIN_HISTORY
    )
    .quantile(0.60)
)


regime_source["q80"] = (
    past_feature
    .expanding(
        min_periods=MIN_HISTORY
    )
    .quantile(0.80)
)


# ============================================================
# 4. CLASSIFY Q1 ~ Q5
# ============================================================

def classify_quintile(row):

    x = row[FEATURE]


    if (
        pd.isna(row["q20"])
        or
        pd.isna(row["q40"])
        or
        pd.isna(row["q60"])
        or
        pd.isna(row["q80"])
    ):

        return np.nan


    if x < row["q20"]:
        return "Q1"

    elif x < row["q40"]:
        return "Q2"

    elif x < row["q60"]:
        return "Q3"

    elif x < row["q80"]:
        return "Q4"

    else:
        return "Q5"


regime_source["bridge_quintile"] = (
    regime_source.apply(
        classify_quintile,
        axis=1
    )
)


# ============================================================
# 5. CHECK DISTRIBUTION
# ============================================================

print("=" * 60)
print("QUINTILE DISTRIBUTION")
print("=" * 60)

print(
    regime_source[
        "bridge_quintile"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# 6. PREPARE QUINTILE INFORMATION
# ============================================================

quintile_info = (
    regime_source[
        [
            "Date",
            FEATURE,

            "q20",
            "q40",
            "q60",
            "q80",

            "bridge_quintile"
        ]
    ]
    .rename(
        columns={
            "Date":
                "feature_date"
        }
    )
)


# ============================================================
# 7. PREPARE M1 / M3 RESULTS
# ============================================================

m1 = results_h3_m1.copy()
m3 = results_h3_m3.copy()


m1["feature_date"] = pd.to_datetime(
    m1["feature_date"]
)

m3["feature_date"] = pd.to_datetime(
    m3["feature_date"]
)


# ============================================================
# 8. ADD QUINTILE INFORMATION
# ============================================================

m1 = (
    m1
    .merge(
        quintile_info,
        on="feature_date",
        how="inner"
    )
)


m3 = (
    m3
    .merge(
        quintile_info,
        on="feature_date",
        how="inner"
    )
)


m1 = m1.dropna(
    subset=["bridge_quintile"]
)


m3 = m3.dropna(
    subset=["bridge_quintile"]
)


# ============================================================
# 9. QUINTILE ANALYSIS
# ============================================================

quintile_results = []


for quintile in [
    "Q1",
    "Q2",
    "Q3",
    "Q4",
    "Q5"
]:

    print()
    print("=" * 60)
    print(
        f"Running {quintile}"
    )
    print("=" * 60)


    # --------------------------------------------------------
    # Select regime
    # --------------------------------------------------------

    m1_q = (
        m1[
            m1["bridge_quintile"]
            ==
            quintile
        ]
        .copy()
    )


    m3_q = (
        m3[
            m3["bridge_quintile"]
            ==
            quintile
        ]
        .copy()
    )


    # ========================================================
    # 10. COMMON DATES
    # ========================================================

    common_dates = sorted(

        set(
            m1_q[
                "feature_date"
            ]
        )

        &

        set(
            m3_q[
                "feature_date"
            ]
        )

    )


    m1_q = (

        m1_q[
            m1_q["feature_date"]
            .isin(
                common_dates
            )
        ]

        .sort_values(
            "feature_date"
        )

        .reset_index(drop=True)

    )


    m3_q = (

        m3_q[
            m3_q["feature_date"]
            .isin(
                common_dates
            )
        ]

        .sort_values(
            "feature_date"
        )

        .reset_index(drop=True)

    )


    # --------------------------------------------------------
    # sample check
    # --------------------------------------------------------

    if len(m1_q) < 10:

        print(
            f"Skip {quintile}: "
            f"n={len(m1_q)}"
        )

        continue


    # ========================================================
    # 11. VERIFY SAME ACTUAL TARGET
    # ========================================================

    if not np.allclose(

        m1_q[
            "actual_volume_change"
        ],

        m3_q[
            "actual_volume_change"
        ]

    ):

        raise ValueError(
            f"Actual target mismatch: "
            f"{quintile}"
        )


    # ========================================================
    # 12. VALUES
    # ========================================================

    y_true = (
        m1_q[
            "actual_volume_change"
        ]
    )


    y_m1 = (
        m1_q[
            "predicted_volume_change"
        ]
    )


    y_m3 = (
        m3_q[
            "predicted_volume_change"
        ]
    )


    # ========================================================
    # 13. MAE
    # ========================================================

    mae_m1 = (
        mean_absolute_error(
            y_true,
            y_m1
        )
    )


    mae_m3 = (
        mean_absolute_error(
            y_true,
            y_m3
        )
    )


    # ========================================================
    # 14. RMSE
    # ========================================================

    rmse_m1 = np.sqrt(

        mean_squared_error(
            y_true,
            y_m1
        )

    )


    rmse_m3 = np.sqrt(

        mean_squared_error(
            y_true,
            y_m3
        )

    )


    # ========================================================
    # 15. R2
    # ========================================================

    r2_m1 = (
        r2_score(
            y_true,
            y_m1
        )
    )


    r2_m3 = (
        r2_score(
            y_true,
            y_m3
        )
    )


    # ========================================================
    # 16. DIRECTION ACCURACY
    # ========================================================

    true_direction = (
        y_true > 0
    ).astype(int)


    direction_m1 = (
        y_m1 > 0
    ).astype(int)


    direction_m3 = (
        y_m3 > 0
    ).astype(int)


    direction_accuracy_m1 = (

        true_direction.values
        ==
        direction_m1.values

    ).mean()


    direction_accuracy_m3 = (

        true_direction.values
        ==
        direction_m3.values

    ).mean()


    # ========================================================
    # 17. FORECAST ERRORS
    # ========================================================

    error_m1 = (
        y_true
        -
        y_m1
    )


    error_m3 = (
        y_true
        -
        y_m3
    )


    # ========================================================
    # 18. DM TEST
    # ========================================================

    dm = dm_test_hac(

        error_m1,

        error_m3,

        horizon=HORIZON

    )


    # ========================================================
    # 19. SAVE
    # ========================================================

    quintile_results.append({

        "quintile":
            quintile,

        "n":
            len(y_true),

        # R2
        "r2_m1":
            r2_m1,

        "r2_m3":
            r2_m3,

        "delta_r2":
            r2_m3
            -
            r2_m1,

        # RMSE
        "rmse_m1":
            rmse_m1,

        "rmse_m3":
            rmse_m3,

        "delta_rmse":
            rmse_m3
            -
            rmse_m1,

        # MAE
        "mae_m1":
            mae_m1,

        "mae_m3":
            mae_m3,

        "delta_mae":
            mae_m3
            -
            mae_m1,

        # direction
        "direction_m1":
            direction_accuracy_m1,

        "direction_m3":
            direction_accuracy_m3,

        "delta_direction":
            direction_accuracy_m3
            -
            direction_accuracy_m1,

        # DM
        "mean_loss_difference":
            dm[
                "mean_loss_difference"
            ],

        "dm_stat":
            dm[
                "dm_stat"
            ],

        "dm_p_value":
            dm[
                "p_value"
            ]
    })


# ============================================================
# 20. RESULT TABLE
# ============================================================

quintile_results = pd.DataFrame(
    quintile_results
)


quintile_results[
    [
        "quintile",
        "n",

        "r2_m1",
        "r2_m3",
        "delta_r2",

        "delta_rmse",
        "delta_mae",

        "direction_m1",
        "direction_m3",
        "delta_direction",

        "dm_stat",
        "dm_p_value"
    ]
]

QUINTILE DISTRIBUTION
bridge_quintile
Q4     258
Q3     253
Q2     241
Q1     231
Q5     227
NaN     30
Name: count, dtype: int64

Running Q1

Running Q2

Running Q3

Running Q4

Running Q5


,quintile,n,r2_m1,r2_m3,delta_r2,delta_rmse,delta_mae,direction_m1,direction_m3,delta_direction,dm_stat,dm_p_value
0,Q1,86,0.528206,0.539119,0.010913,-0.004009,-0.006468,0.895349,0.872093,-0.023256,0.424581,0.671142
1,Q2,67,0.431098,0.444646,0.013547,-0.004617,-0.009353,0.686567,0.746269,0.059701,0.552656,0.580499
2,Q3,76,0.258943,0.228685,-0.030258,0.008782,0.010653,0.605263,0.592105,-0.013158,-1.058882,0.289653
3,Q4,69,-0.061594,-0.023327,0.038267,-0.008623,-0.012819,0.768116,0.724638,-0.043478,2.012016,0.044218
4,Q5,67,0.095229,0.122729,0.027500,-0.007010,-0.001194,0.686567,0.716418,0.029851,0.933780,0.350417


In [85]:
quintile_results[
    [
        "quintile",
        "n",
        "delta_r2",
        "delta_rmse",
        "delta_mae",
        "dm_p_value"
    ]
]

,quintile,n,delta_r2,delta_rmse,delta_mae,dm_p_value
0,Q1,86,0.010913,-0.004009,-0.006468,0.671142
1,Q2,67,0.013547,-0.004617,-0.009353,0.580499
2,Q3,76,-0.030258,0.008782,0.010653,0.289653
3,Q4,69,0.038267,-0.008623,-0.012819,0.044218
4,Q5,67,0.027500,-0.007010,-0.001194,0.350417


In [ ]:
# Use the common M1/M3 forecast dates and classifications from the preceding analyses.
comparison_dates = pd.Index(results_h3_m1["feature_date"]).intersection(
    pd.Index(results_h3_m3["feature_date"])
)
three_regime_predictions = regime_info.loc[
    regime_info["feature_date"].isin(comparison_dates),
    ["feature_date", "bridge_regime"],
].copy()
quantile_predictions = quintile_info.loc[
    quintile_info["feature_date"].isin(comparison_dates),
    ["feature_date", "bridge_quintile"],
].rename(columns={"bridge_quintile": "quintile"})

# Dates in the High regime
high_dates = set(
    three_regime_predictions.loc[
        three_regime_predictions["bridge_regime"] == "High",
        "feature_date"
    ]
)

# Dates in Q4
q4_dates = set(
    quantile_predictions.loc[
        quantile_predictions["quintile"] == "Q4",
        "feature_date"
    ]
)

print("High n:", len(high_dates))
print("Q4 n:", len(q4_dates))

print("Common:", len(high_dates & q4_dates))
print("High only:", len(high_dates - q4_dates))
print("Q4 only:", len(q4_dates - high_dates))

print("Exactly same dates:", high_dates == q4_dates)

In [87]:
from statsmodels.stats.multitest import multipletests

p_values = quintile_results["dm_p_value"].values

# Benjamini-Hochberg FDR
reject_fdr, p_fdr, _, _ = multipletests(
    p_values,
    alpha=0.05,
    method="fdr_bh"
)

# Bonferroni
reject_bonf, p_bonf, _, _ = multipletests(
    p_values,
    alpha=0.05,
    method="bonferroni"
)

quintile_results["dm_p_fdr"] = p_fdr
quintile_results["dm_p_bonf"] = p_bonf
quintile_results["sig_fdr"] = reject_fdr
quintile_results["sig_bonf"] = reject_bonf

quintile_results[
    [
        "quintile",
        "delta_r2",
        "dm_p_value",
        "dm_p_fdr",
        "dm_p_bonf",
        "sig_fdr",
        "sig_bonf"
    ]
]

,quintile,delta_r2,dm_p_value,dm_p_fdr,dm_p_bonf,sig_fdr,sig_bonf
0,Q1,0.010913,0.671142,0.671142,1.000000,False,False
1,Q2,0.013547,0.580499,0.671142,1.000000,False,False
2,Q3,-0.030258,0.289653,0.584029,1.000000,False,False
3,Q4,0.038267,0.044218,0.221091,0.221091,False,False
4,Q5,0.027500,0.350417,0.584029,1.000000,False,False


## 5. RF, XGBoost, and LightGBM comparisons

In [90]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# ============================================================
# 1. SETTINGS
# ============================================================

TRAIN_START = pd.Timestamp("2023-08-01")
TEST_START  = pd.Timestamp("2025-08-01")
TEST_END    = pd.Timestamp("2026-07-31")

HORIZON = 3
RETRAIN_DAYS = 7

VOLUME_COL = "volume"
TARGET_COL = "future_volume_change"

RANDOM_STATE = 42


# ============================================================
# 2. REGIME SETTINGS
# Q4 = 60% ~ 80%
# ============================================================

REGIME_FEATURE = "bridge_volume_momentum_7d"

LOW_Q  = 0.60
HIGH_Q = 0.80


# ============================================================
# 3. MODEL FACTORY
# ============================================================

def make_model(model_type):

    if model_type == "RF":

        return RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    elif model_type == "XGB":

        return XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    elif model_type == "LGBM":

        return LGBMRegressor(
            n_estimators=300,
            max_depth=-1,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1
        )

    else:

        raise ValueError(
            "model_type must be RF, XGB, or LGBM"
        )


# ============================================================
# 4. GENERIC FORWARD FORECAST FUNCTION
# ============================================================

def run_forward_model(
    input_data,
    model_type,
    model_name
):

    data = input_data.copy()


    # --------------------------------------------------------
    # Date
    # --------------------------------------------------------

    if data.index.name == "Date":
        data = data.reset_index()

    data["Date"] = pd.to_datetime(
        data["Date"]
    )

    data = (
        data
        .sort_values("Date")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    data[VOLUME_COL] = pd.to_numeric(
        data[VOLUME_COL],
        errors="coerce"
    )

    data.loc[
        data[VOLUME_COL] <= 0,
        VOLUME_COL
    ] = np.nan


    data[TARGET_COL] = (
        np.log(
            data[VOLUME_COL].shift(-HORIZON)
        )
        -
        np.log(
            data[VOLUME_COL]
        )
    )


    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    exclude_cols = [
        "Date",
        TARGET_COL,
        "target",
        "future_return_d"
    ]


    feature_cols = [
        c
        for c in data.columns
        if c not in exclude_cols
    ]


    for c in feature_cols:

        data[c] = pd.to_numeric(
            data[c],
            errors="coerce"
        )


    data = data.replace(
        [np.inf, -np.inf],
        np.nan
    )


    data = (
        data
        .dropna(
            subset=
            feature_cols
            +
            [TARGET_COL]
        )
        .reset_index(drop=True)
    )


    data = data[
        (data["Date"] >= TRAIN_START)
        &
        (data["Date"] <= TEST_END)
    ].copy()


    # --------------------------------------------------------
    # Test indices
    # --------------------------------------------------------

    test_indices = data.index[
        (data["Date"] >= TEST_START)
        &
        (data["Date"] <= TEST_END)
    ].tolist()


    predictions = []

    model = None
    last_retrain_date = None


    # ========================================================
    # Forward loop
    # ========================================================

    for idx in test_indices:

        current_date = data.loc[
            idx,
            "Date"
        ]


        need_retrain = (
            model is None
            or
            (
                current_date
                -
                last_retrain_date
            ).days
            >= RETRAIN_DAYS
        )


        if need_retrain:

            observable_cutoff = (
                current_date
                -
                pd.Timedelta(
                    days=HORIZON
                )
            )


            train = data[
                (data["Date"] >= TRAIN_START)
                &
                (data["Date"] <= observable_cutoff)
            ].copy()


            X_train = train[
                feature_cols
            ]

            y_train = train[
                TARGET_COL
            ]


            model = make_model(
                model_type
            )


            model.fit(
                X_train,
                y_train
            )


            last_retrain_date = (
                current_date
            )


        X_today = data.loc[
            [idx],
            feature_cols
        ]


        pred = model.predict(
            X_today
        )[0]


        actual = float(
            data.loc[
                idx,
                TARGET_COL
            ]
        )


        predictions.append({

            "feature_date":
                current_date,

            "actual_volume_change":
                actual,

            "predicted_volume_change":
                pred

        })


    results = pd.DataFrame(
        predictions
    )


    return (
        results,
        feature_cols
    )


# ============================================================
# 5. BUILD EXPANDING Q4 REGIME
# ============================================================

regime_data = data_m4.copy()


if regime_data.index.name == "Date":
    regime_data = regime_data.reset_index()


regime_data["Date"] = pd.to_datetime(
    regime_data["Date"]
)


regime_data = (
    regime_data
    .sort_values("Date")
    .reset_index(drop=True)
)


past = (
    regime_data[
        REGIME_FEATURE
    ]
    .shift(1)
)


regime_data["q60"] = (
    past
    .expanding(
        min_periods=30
    )
    .quantile(
        LOW_Q
    )
)


regime_data["q80"] = (
    past
    .expanding(
        min_periods=30
    )
    .quantile(
        HIGH_Q
    )
)


regime_data["Q4"] = (

    (
        regime_data[
            REGIME_FEATURE
        ]
        >=
        regime_data["q60"]
    )

    &

    (
        regime_data[
            REGIME_FEATURE
        ]
        <
        regime_data["q80"]
    )

)


q4_dates = set(

    regime_data.loc[
        regime_data["Q4"],
        "Date"
    ]

)


# ============================================================
# 6. RUN MODEL ROBUSTNESS
# ============================================================

model_results = []


for model_type in [
    "RF",
    "XGB",
    "LGBM"
]:

    print()
    print("=" * 70)
    print(
        f"MODEL: {model_type}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # M1
    # --------------------------------------------------------

    m1_results, _ = (
        run_forward_model(
            data_m1,
            model_type=model_type,
            model_name="M1"
        )
    )


    # --------------------------------------------------------
    # M3
    # --------------------------------------------------------

    m3_results, _ = (
        run_forward_model(
            data_m3,
            model_type=model_type,
            model_name="M3"
        )
    )


    # ========================================================
    # 7. COMMON DATES + Q4 ONLY
    # ========================================================

    compare = (

        m1_results

        .merge(
            m3_results,
            on="feature_date",
            suffixes=(
                "_m1",
                "_m3"
            )
        )

    )


    compare = compare[
        compare[
            "feature_date"
        ].isin(
            q4_dates
        )
    ].copy()


    compare = (
        compare
        .sort_values(
            "feature_date"
        )
        .reset_index(drop=True)
    )


    # ========================================================
    # 8. TARGET CHECK
    # ========================================================

    if not np.allclose(

        compare[
            "actual_volume_change_m1"
        ],

        compare[
            "actual_volume_change_m3"
        ]

    ):

        raise ValueError(
            f"Actual target mismatch: "
            f"{model_type}"
        )


    y_true = compare[
        "actual_volume_change_m1"
    ]

    y_m1 = compare[
        "predicted_volume_change_m1"
    ]

    y_m3 = compare[
        "predicted_volume_change_m3"
    ]


    # ========================================================
    # 9. METRICS
    # ========================================================

    r2_m1 = r2_score(
        y_true,
        y_m1
    )

    r2_m3 = r2_score(
        y_true,
        y_m3
    )


    rmse_m1 = np.sqrt(
        mean_squared_error(
            y_true,
            y_m1
        )
    )

    rmse_m3 = np.sqrt(
        mean_squared_error(
            y_true,
            y_m3
        )
    )


    mae_m1 = mean_absolute_error(
        y_true,
        y_m1
    )

    mae_m3 = mean_absolute_error(
        y_true,
        y_m3
    )


    # ========================================================
    # 10. DM TEST
    # ========================================================

    error_m1 = (
        y_true
        -
        y_m1
    )

    error_m3 = (
        y_true
        -
        y_m3
    )


    dm = dm_test_hac(
        error_m1,
        error_m3,
        horizon=HORIZON
    )


    # ========================================================
    # 11. SAVE
    # ========================================================

    model_results.append({

        "model":
            model_type,

        "n":
            len(compare),

        "r2_m1":
            r2_m1,

        "r2_m3":
            r2_m3,

        "delta_r2":
            r2_m3
            -
            r2_m1,

        "rmse_m1":
            rmse_m1,

        "rmse_m3":
            rmse_m3,

        "delta_rmse":
            rmse_m3
            -
            rmse_m1,

        "mae_m1":
            mae_m1,

        "mae_m3":
            mae_m3,

        "delta_mae":
            mae_m3
            -
            mae_m1,

        "dm_stat":
            dm[
                "dm_stat"
            ],

        "dm_p_value":
            dm[
                "p_value"
            ]

    })


# ============================================================
# 12. FINAL TABLE
# ============================================================

model_robustness = pd.DataFrame(
    model_results
)


model_robustness[
    [
        "model",
        "n",

        "r2_m1",
        "r2_m3",
        "delta_r2",

        "delta_rmse",
        "delta_mae",

        "dm_stat",
        "dm_p_value"
    ]
]


MODEL: RF

MODEL: XGB

MODEL: LGBM


,model,n,r2_m1,r2_m3,delta_r2,delta_rmse,delta_mae,dm_stat,dm_p_value
0,RF,69,-0.061594,-0.023327,0.038267,-0.008623,-0.012819,2.012016,0.044218
1,XGB,69,-0.178451,-0.127220,0.051230,-0.010979,-0.017618,1.542836,0.122871
2,LGBM,69,-0.133443,-0.168088,-0.034645,0.007431,-0.000341,-0.629038,0.529324


Resolved 57 packages in 1.83s                                        
Installed 1 package in 6ms                                       
 + lightgbm==4.7.0


In [91]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. SETTINGS
# ============================================================

HORIZON = 3

REGIME_FEATURE = "bridge_volume_momentum_7d"

LOW_Q = 0.60
HIGH_Q = 0.80

MIN_HISTORY = 30


# ============================================================
# 2. PREPARE DATA
# ============================================================

m1_data = data_m1.copy()
m3_data = data_m3.copy()


# ------------------------------------------------------------
# Date handling
# ------------------------------------------------------------

for df in [m1_data, m3_data]:

    if df.index.name == "Date":
        df.reset_index(inplace=True)

    df["Date"] = pd.to_datetime(
        df["Date"]
    )


m1_data = (
    m1_data
    .sort_values("Date")
    .reset_index(drop=True)
)


m3_data = (
    m3_data
    .sort_values("Date")
    .reset_index(drop=True)
)


# ============================================================
# 3. CREATE M3 ABLATION DATA
# ============================================================
#
# Remove momentum from predictive features
#
# IMPORTANT:
# We still use momentum later to DEFINE Q4 regime,
# but the ablation model itself cannot see this feature.
# ============================================================

if REGIME_FEATURE not in m3_data.columns:

    raise ValueError(
        f"{REGIME_FEATURE} not found in data_m3"
    )


m3_ablation_data = (
    m3_data
    .drop(
        columns=[
            REGIME_FEATURE
        ]
    )
    .copy()
)


print(
    "M1 features:",
    len(m1_data.columns)
)

print(
    "M3 features:",
    len(m3_data.columns)
)

print(
    "M3 Ablation features:",
    len(m3_ablation_data.columns)
)


# ============================================================
# 4. RUN FORECASTS
# ============================================================

results_m1, _, summary_m1 = (
    test_volume(
        m1_data,
        horizon=HORIZON,
        model_name="M1 Market"
    )
)


results_m3, _, summary_m3 = (
    test_volume(
        m3_data,
        horizon=HORIZON,
        model_name="M3 Engineered Bridge"
    )
)


results_ablation, _, summary_ablation = (
    test_volume(
        m3_ablation_data,
        horizon=HORIZON,
        model_name="M3 Ablation"
    )
)


# ============================================================
# 5. BUILD Q4 REGIME
# ============================================================
#
# Q4 =
#
# 60th percentile <= Momentum < 80th percentile
#
# Expanding thresholds use only past observations
# ============================================================

regime_source = (
    m3_data[
        [
            "Date",
            REGIME_FEATURE
        ]
    ]
    .copy()
)


regime_source = (
    regime_source
    .sort_values("Date")
    .reset_index(drop=True)
)


past_momentum = (
    regime_source[
        REGIME_FEATURE
    ]
    .shift(1)
)


regime_source["q60"] = (

    past_momentum
    .expanding(
        min_periods=MIN_HISTORY
    )
    .quantile(
        LOW_Q
    )

)


regime_source["q80"] = (

    past_momentum
    .expanding(
        min_periods=MIN_HISTORY
    )
    .quantile(
        HIGH_Q
    )

)


regime_source["is_q4"] = (

    (
        regime_source[
            REGIME_FEATURE
        ]
        >=
        regime_source[
            "q60"
        ]
    )

    &

    (
        regime_source[
            REGIME_FEATURE
        ]
        <
        regime_source[
            "q80"
        ]
    )

)


q4_dates = set(

    regime_source.loc[
        regime_source[
            "is_q4"
        ],
        "Date"
    ]

)


print(
    "Number of Q4 dates:",
    len(q4_dates)
)


# ============================================================
# 6. PREPARE FORECAST RESULTS
# ============================================================

def prepare_result(
    df,
    model_name
):

    temp = df[
        [
            "feature_date",
            "actual_volume_change",
            "predicted_volume_change"
        ]
    ].copy()


    temp["feature_date"] = (
        pd.to_datetime(
            temp["feature_date"]
        )
    )


    temp = temp.rename(
        columns={
            "actual_volume_change":
                f"actual_{model_name}",

            "predicted_volume_change":
                f"pred_{model_name}"
        }
    )


    return temp


m1_r = prepare_result(
    results_m1,
    "m1"
)

m3_r = prepare_result(
    results_m3,
    "m3"
)

abl_r = prepare_result(
    results_ablation,
    "ablation"
)


# ============================================================
# 7. MERGE ALL THREE MODELS
# ============================================================

compare = (

    m1_r

    .merge(
        m3_r,
        on="feature_date",
        how="inner"
    )

    .merge(
        abl_r,
        on="feature_date",
        how="inner"
    )

)


# ============================================================
# 8. KEEP Q4 ONLY
# ============================================================

compare = (
    compare[
        compare[
            "feature_date"
        ].isin(
            q4_dates
        )
    ]
    .sort_values(
        "feature_date"
    )
    .reset_index(drop=True)
)


print(
    "Q4 common test observations:",
    len(compare)
)


# ============================================================
# 9. CHECK SAME ACTUAL TARGET
# ============================================================

if not np.allclose(
    compare["actual_m1"],
    compare["actual_m3"]
):

    raise ValueError(
        "M1 and M3 targets differ."
    )


if not np.allclose(
    compare["actual_m1"],
    compare["actual_ablation"]
):

    raise ValueError(
        "M1 and Ablation targets differ."
    )


y_true = (
    compare[
        "actual_m1"
    ]
)


# ============================================================
# 10. METRIC FUNCTION
# ============================================================

from eth_bridge.evaluation import calculate_metrics


# ============================================================
# 11. METRICS FOR M1 / M3 / ABLATION
# ============================================================

metrics_m1 = (
    calculate_metrics(
        y_true,
        compare[
            "pred_m1"
        ]
    )
)


metrics_m3 = (
    calculate_metrics(
        y_true,
        compare[
            "pred_m3"
        ]
    )
)


metrics_ablation = (
    calculate_metrics(
        y_true,
        compare[
            "pred_ablation"
        ]
    )
)


# ============================================================
# 12. DM TEST:
# M1 vs M3
# ============================================================

error_m1 = (
    y_true
    -
    compare[
        "pred_m1"
    ]
)


error_m3 = (
    y_true
    -
    compare[
        "pred_m3"
    ]
)


error_ablation = (
    y_true
    -
    compare[
        "pred_ablation"
    ]
)


dm_m1_vs_m3 = (
    dm_test_hac(
        error_m1,
        error_m3,
        horizon=HORIZON
    )
)


# ============================================================
# 13. DM TEST:
# M1 vs ABLATION
# ============================================================

dm_m1_vs_ablation = (
    dm_test_hac(
        error_m1,
        error_ablation,
        horizon=HORIZON
    )
)


# ============================================================
# 14. DM TEST:
# FULL M3 vs ABLATION
# ============================================================

dm_m3_vs_ablation = (
    dm_test_hac(
        error_m3,
        error_ablation,
        horizon=HORIZON
    )
)


# ============================================================
# 15. FINAL PERFORMANCE TABLE
# ============================================================

ablation_results = pd.DataFrame([

    {
        "model":
            "M1 Market",

        "n":
            len(compare),

        **metrics_m1
    },


    {
        "model":
            "M3 Full Engineered",

        "n":
            len(compare),

        **metrics_m3
    },


    {
        "model":
            "M3 Without Momentum",

        "n":
            len(compare),

        **metrics_ablation
    }

])


ablation_results

M1 features: 104
M3 features: 118
M3 Ablation features: 117

M1 Market | ETH Volume t+3
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-29 | n=729
Retrain: 2025-08-08 | Train through: 2025-08-05 | n=736
Retrain: 2025-08-15 | Train through: 2025-08-12 | n=743
Retrain: 2025-08-22 | Train through: 2025-08-19 | n=750
Retrain: 2025-08-29 | Train through: 2025-08-26 | n=757
Retrain: 2025-09-05 | Train through: 2025-09-02 | n=764
Retrain: 2025-09-12 | Train through: 2025-09-09 | n=771
Retrain: 2025-09-19 | Train through: 2025-09-16 | n=778
Retrain: 2025-09-26 | Train through: 2025-09-23 | n=785
Retrain: 2025-10-03 | Train through: 2025-09-30 | n=792
Retrain: 2025-10-10 | Train through: 2025-10-07 | n=799
Retrain: 2025-10-17 | Train through: 2025-10-14 | n=806
Retrain: 2025-10-24 | Train through: 2025-10-21 | n=813
Retrain: 2025-10-31 | Train through: 2025-10-28 | n=820
Retrain: 2025-11-07 | Train through: 2025-11-04 

,model,n,mae,rmse,r2,direction_accuracy
0,M1 Market,69,0.363291,0.474113,-0.061594,0.768116
1,M3 Full Engineered,69,0.350472,0.465490,-0.023327,0.724638
2,M3 Without Momentum,69,0.360804,0.472563,-0.054662,0.724638


In [ ]:
baseline_r2 = (
    metrics_m1["r2"]
)

baseline_rmse = (
    metrics_m1["rmse"]
)

baseline_mae = (
    metrics_m1["mae"]
)


ablation_comparison = pd.DataFrame([

    {
        "model":
            "M3 Full",

        "delta_r2":
            metrics_m3["r2"]
            -
            baseline_r2,

        "delta_rmse":
            metrics_m3["rmse"]
            -
            baseline_rmse,

        "delta_mae":
            metrics_m3["mae"]
            -
            baseline_mae,

        "dm_p_vs_m1":
            dm_m1_vs_m3[
                "p_value"
            ]
    },


    {
        "model":
            "M3 Without Momentum",

        "delta_r2":
            metrics_ablation["r2"]
            -
            baseline_r2,

        "delta_rmse":
            metrics_ablation["rmse"]
            -
            baseline_rmse,

        "delta_mae":
            metrics_ablation["mae"]
            -
            baseline_mae,

        "dm_p_vs_m1":
            dm_m1_vs_ablation[
                "p_value"
            ]
    }

])


ablation_comparison
# table 7

,model,delta_r2,delta_rmse,delta_mae,dm_p_vs_m1
0,M3 Full,0.038267,-0.008623,-0.012819,0.044218
1,M3 Without Momentum,0.006931,-0.001550,-0.002486,0.698724


In [93]:
print(
    "M3 Full vs M3 Without Momentum"
)

print(
    dm_m3_vs_ablation
)

M3 Full vs M3 Without Momentum
{'mean_loss_difference': np.float64(-0.0066349989162624926), 'dm_stat': np.float64(-2.542966098216092), 'p_value': np.float64(0.010991589317879358)}


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. SETTINGS
# ============================================================

HORIZON = 3

LOW_Q  = 0.60
HIGH_Q = 0.80

MIN_HISTORY = 30

MOMENTUM_FEATURE = "bridge_volume_momentum_7d"


# ============================================================
# 2. INDEPENDENT REGIME VARIABLES
# ============================================================
#
# IMPORTANT:
#
# Intentionally exclude bridge_volume_momentum_7d
#
# Define regimes using bridge activity variables
# independent of momentum
# ============================================================

REGIME_FEATURES = [

    "gross_volume_usd",

    "total_tx_count",

    "total_outflow_usd",

    "total_inflow_usd"
]


# ============================================================
# 3. PREPARE MODEL DATA
# ============================================================

m1_data = data_m1.copy()
m3_data = data_m3.copy()


# Date handling
for df in [m1_data, m3_data]:

    if df.index.name == "Date":
        df.reset_index(inplace=True)

    df["Date"] = pd.to_datetime(
        df["Date"]
    )


m1_data = (
    m1_data
    .sort_values("Date")
    .reset_index(drop=True)
)


m3_data = (
    m3_data
    .sort_values("Date")
    .reset_index(drop=True)
)


# ============================================================
# 4. CREATE ABLATION DATA
# ============================================================
#
# M3 without bridge_volume_momentum_7d
# ============================================================

if MOMENTUM_FEATURE not in m3_data.columns:

    raise ValueError(
        f"{MOMENTUM_FEATURE} not found in data_m3"
    )


m3_ablation_data = (
    m3_data
    .drop(
        columns=[
            MOMENTUM_FEATURE
        ]
    )
    .copy()
)


# ============================================================
# 5. RUN FORECASTS ONCE
# ============================================================
#
# No need to retrain models for each regime.
#
# Generate the full set of forward forecasts, then
# subset results by independent regimes
# ============================================================

results_m1, _, summary_m1 = (
    test_volume(
        m1_data,
        horizon=HORIZON,
        model_name="M1 Market"
    )
)


results_m3, _, summary_m3 = (
    test_volume(
        m3_data,
        horizon=HORIZON,
        model_name="M3 Full Engineered"
    )
)


results_ablation, _, summary_ablation = (
    test_volume(
        m3_ablation_data,
        horizon=HORIZON,
        model_name="M3 Without Momentum"
    )
)


# ============================================================
# 6. PREPARE FORECAST RESULTS
# ============================================================

def prepare_predictions(
    results,
    suffix
):

    temp = results[
        [
            "feature_date",
            "actual_volume_change",
            "predicted_volume_change"
        ]
    ].copy()


    temp["feature_date"] = pd.to_datetime(
        temp["feature_date"]
    )


    temp = temp.rename(
        columns={

            "actual_volume_change":
                f"actual_{suffix}",

            "predicted_volume_change":
                f"pred_{suffix}"
        }
    )


    return temp


m1_pred = prepare_predictions(
    results_m1,
    "m1"
)


m3_pred = prepare_predictions(
    results_m3,
    "m3"
)


abl_pred = prepare_predictions(
    results_ablation,
    "ablation"
)


# ============================================================
# 7. MERGE ALL MODEL PREDICTIONS
# ============================================================

all_predictions = (

    m1_pred

    .merge(
        m3_pred,
        on="feature_date",
        how="inner"
    )

    .merge(
        abl_pred,
        on="feature_date",
        how="inner"
    )

    .sort_values(
        "feature_date"
    )

    .reset_index(drop=True)

)


# ============================================================
# 8. VERIFY SAME ACTUAL TARGET
# ============================================================

if not np.allclose(
    all_predictions["actual_m1"],
    all_predictions["actual_m3"]
):

    raise ValueError(
        "M1 and M3 actual targets differ."
    )


if not np.allclose(
    all_predictions["actual_m1"],
    all_predictions["actual_ablation"]
):

    raise ValueError(
        "M1 and Ablation actual targets differ."
    )


# ============================================================
# 9. PREPARE REGIME SOURCE
# ============================================================

regime_source = data_m4.copy()


if regime_source.index.name == "Date":
    regime_source = regime_source.reset_index()


regime_source["Date"] = pd.to_datetime(
    regime_source["Date"]
)


regime_source = (
    regime_source
    .sort_values("Date")
    .reset_index(drop=True)
)


# Check regime variables
missing_regime_features = [

    f for f in REGIME_FEATURES

    if f not in regime_source.columns
]


if missing_regime_features:

    raise ValueError(
        f"Missing regime features: "
        f"{missing_regime_features}"
    )


# ============================================================
# 10. METRIC FUNCTION
# ============================================================

from eth_bridge.evaluation import calculate_metrics


# ============================================================
# 11. STORAGE
# ============================================================

independent_regime_results = []


# ============================================================
# 12. LOOP THROUGH INDEPENDENT REGIME FEATURES
# ============================================================

for regime_feature in REGIME_FEATURES:

    print()
    print("=" * 80)
    print(
        f"REGIME FEATURE: {regime_feature}"
    )
    print("=" * 80)


    # --------------------------------------------------------
    # Regime data
    # --------------------------------------------------------

    temp = regime_source[
        [
            "Date",
            regime_feature
        ]
    ].copy()


    temp[regime_feature] = pd.to_numeric(
        temp[regime_feature],
        errors="coerce"
    )


    # ========================================================
    # 13. EXPANDING Q60 / Q80
    # ========================================================
    #
    # shift(1):
    #
    # Today's thresholds are based ONLY on past information.
    # ========================================================

    past = (
        temp[regime_feature]
        .shift(1)
    )


    temp["q60"] = (

        past
        .expanding(
            min_periods=MIN_HISTORY
        )
        .quantile(
            LOW_Q
        )

    )


    temp["q80"] = (

        past
        .expanding(
            min_periods=MIN_HISTORY
        )
        .quantile(
            HIGH_Q
        )

    )


    # ========================================================
    # 14. DEFINE Q4
    # ========================================================

    temp["is_q4"] = (

        (
            temp[regime_feature]
            >=
            temp["q60"]
        )

        &

        (
            temp[regime_feature]
            <
            temp["q80"]
        )

    )


    # undefined early period
    temp.loc[
        temp["q60"].isna()
        |
        temp["q80"].isna(),
        "is_q4"
    ] = False


    q4_dates = set(

        temp.loc[
            temp["is_q4"],
            "Date"
        ]

    )


    # ========================================================
    # 15. SELECT Q4 FORECASTS
    # ========================================================

    compare = (

        all_predictions[
            all_predictions[
                "feature_date"
            ].isin(
                q4_dates
            )
        ]

        .copy()

        .sort_values(
            "feature_date"
        )

        .reset_index(drop=True)

    )


    n = len(compare)


    print(
        f"Q4 test observations: {n}"
    )


    if n < 20:

        print(
            "Too few observations. Skip."
        )

        continue


    # ========================================================
    # 16. VALUES
    # ========================================================

    y_true = (
        compare[
            "actual_m1"
        ]
    )


    pred_m1 = (
        compare[
            "pred_m1"
        ]
    )


    pred_m3 = (
        compare[
            "pred_m3"
        ]
    )


    pred_ablation = (
        compare[
            "pred_ablation"
        ]
    )


    # ========================================================
    # 17. METRICS
    # ========================================================

    metrics_m1 = calculate_metrics(
        y_true,
        pred_m1
    )


    metrics_m3 = calculate_metrics(
        y_true,
        pred_m3
    )


    metrics_ablation = calculate_metrics(
        y_true,
        pred_ablation
    )


    # ========================================================
    # 18. ERRORS
    # ========================================================

    error_m1 = (
        y_true
        -
        pred_m1
    )


    error_m3 = (
        y_true
        -
        pred_m3
    )


    error_ablation = (
        y_true
        -
        pred_ablation
    )


    # ========================================================
    # 19. DM TESTS
    # ========================================================

    # M1 vs Full M3
    dm_m1_m3 = dm_test_hac(

        error_m1,
        error_m3,

        horizon=HORIZON
    )


    # M1 vs Ablation
    dm_m1_ablation = dm_test_hac(

        error_m1,
        error_ablation,

        horizon=HORIZON
    )


    # Full M3 vs Ablation
    dm_m3_ablation = dm_test_hac(

        error_m3,
        error_ablation,

        horizon=HORIZON
    )


    # ========================================================
    # 20. STORE RESULT
    # ========================================================

    independent_regime_results.append({

        "regime_feature":
            regime_feature,

        "n":
            n,

        # ----------------------------------------------------
        # M1
        # ----------------------------------------------------

        "r2_m1":
            metrics_m1["r2"],

        "rmse_m1":
            metrics_m1["rmse"],

        "mae_m1":
            metrics_m1["mae"],


        # ----------------------------------------------------
        # M3 FULL
        # ----------------------------------------------------

        "r2_m3":
            metrics_m3["r2"],

        "rmse_m3":
            metrics_m3["rmse"],

        "mae_m3":
            metrics_m3["mae"],


        # ----------------------------------------------------
        # M3 WITHOUT MOMENTUM
        # ----------------------------------------------------

        "r2_ablation":
            metrics_ablation["r2"],

        "rmse_ablation":
            metrics_ablation["rmse"],

        "mae_ablation":
            metrics_ablation["mae"],


        # ----------------------------------------------------
        # M3 FULL vs M1
        # ----------------------------------------------------

        "delta_r2_m3":
            metrics_m3["r2"]
            -
            metrics_m1["r2"],

        "delta_rmse_m3":
            metrics_m3["rmse"]
            -
            metrics_m1["rmse"],

        "delta_mae_m3":
            metrics_m3["mae"]
            -
            metrics_m1["mae"],

        "p_m1_vs_m3":
            dm_m1_m3[
                "p_value"
            ],


        # ----------------------------------------------------
        # ABLATION vs M1
        # ----------------------------------------------------

        "delta_r2_ablation":
            metrics_ablation["r2"]
            -
            metrics_m1["r2"],

        "delta_rmse_ablation":
            metrics_ablation["rmse"]
            -
            metrics_m1["rmse"],

        "delta_mae_ablation":
            metrics_ablation["mae"]
            -
            metrics_m1["mae"],

        "p_m1_vs_ablation":
            dm_m1_ablation[
                "p_value"
            ],


        # ----------------------------------------------------
        # M3 FULL vs ABLATION
        # ----------------------------------------------------

        "p_m3_vs_ablation":
            dm_m3_ablation[
                "p_value"
            ],

        "direction_m1":
            metrics_m1[
                "direction_accuracy"
            ],

        "direction_m3":
            metrics_m3[
                "direction_accuracy"
            ],

        "direction_ablation":
            metrics_ablation[
                "direction_accuracy"
            ]
    })


# ============================================================
# 21. FINAL DATAFRAME
# ============================================================

independent_regime_results = pd.DataFrame(
    independent_regime_results
)


# ============================================================
# 22. MAIN RESULT TABLE
# ============================================================

independent_regime_results[
    [
        "regime_feature",
        "n",

        "r2_m1",
        "r2_m3",
        "r2_ablation",

        "delta_r2_m3",
        "delta_r2_ablation",

        "delta_rmse_m3",
        "delta_rmse_ablation",

        "p_m1_vs_m3",
        "p_m1_vs_ablation",
        "p_m3_vs_ablation"
    ]
]
# Table 9


M1 Market | ETH Volume t+3
Number of features: 101
Data period: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Retrain: 2025-08-01 | Train through: 2025-07-29 | n=729
Retrain: 2025-08-08 | Train through: 2025-08-05 | n=736
Retrain: 2025-08-15 | Train through: 2025-08-12 | n=743
Retrain: 2025-08-22 | Train through: 2025-08-19 | n=750
Retrain: 2025-08-29 | Train through: 2025-08-26 | n=757
Retrain: 2025-09-05 | Train through: 2025-09-02 | n=764
Retrain: 2025-09-12 | Train through: 2025-09-09 | n=771
Retrain: 2025-09-19 | Train through: 2025-09-16 | n=778
Retrain: 2025-09-26 | Train through: 2025-09-23 | n=785
Retrain: 2025-10-03 | Train through: 2025-09-30 | n=792
Retrain: 2025-10-10 | Train through: 2025-10-07 | n=799
Retrain: 2025-10-17 | Train through: 2025-10-14 | n=806
Retrain: 2025-10-24 | Train through: 2025-10-21 | n=813
Retrain: 2025-10-31 | Train through: 2025-10-28 | n=820
Retrain: 2025-11-07 | Train through: 2025-11-04 | n=827
Retrain: 2025-11-14 | Train through: 2025-11-11 | n=

,regime_feature,n,r2_m1,r2_m3,r2_ablation,delta_r2_m3,delta_r2_ablation,delta_rmse_m3,delta_rmse_ablation,p_m1_vs_m3,p_m1_vs_ablation,p_m3_vs_ablation
0,gross_volume_usd,49,0.378902,0.380057,0.378215,0.001155,-0.000687,-0.000404,0.000240,0.962360,0.944311,0.929658
1,total_tx_count,38,0.012908,0.103135,0.041723,0.090227,0.028814,-0.018658,-0.005862,0.007911,0.231100,0.031218
2,total_outflow_usd,68,0.245649,0.275267,0.271166,0.029618,0.025517,-0.009210,-0.007924,0.127716,0.004231,0.802959
3,total_inflow_usd,53,0.419206,0.433692,0.428115,0.014486,0.008909,-0.005333,-0.003272,0.608176,0.549117,0.805355


In [95]:
valid = independent_regime_results[
    "p_m1_vs_m3"
].notna()


_, corrected_p, _, _ = multipletests(
    independent_regime_results.loc[
        valid,
        "p_m1_vs_m3"
    ],
    method="fdr_bh"
)


independent_regime_results.loc[
    valid,
    "p_m1_vs_m3_fdr"
] = corrected_p


independent_regime_results[
    [
        "regime_feature",
        "p_m1_vs_m3",
        "p_m1_vs_m3_fdr"
    ]
]

,regime_feature,p_m1_vs_m3,p_m1_vs_m3_fdr
0,gross_volume_usd,0.962360,0.962360
1,total_tx_count,0.007911,0.031644
2,total_outflow_usd,0.127716,0.255432
3,total_inflow_usd,0.608176,0.810902
